# D0 · held-out export (pilot D) · CPU

**What it does.** It rebuilds pilot D's held-out task pool from public GitHub (the commits listed in
`pilot_d/manifest.json`) and validates every task in this Kaggle image with the harness's own `verify_task`: the gold
patch must pass and the empty patch must fail. It then fetches issue text from the GitHub API for valid tasks, computes
F2P/P2P, gold locations and verified reproduction tests, and writes released-rule graphs and substitute embeddings.
Everything goes into the `docs/PROBE_INPUT.md` (paper/main) **Level A** layout.

| | |
|---|---|
| Accelerator | **CPU** (no GPU quota) |
| Internet | **ON**: GitHub clone + API, PyPI (test deps), Hugging Face (embedding model `BAAI/bge-small-en-v1.5`, safetensors, no remote code) |
| Environment | the official starter's **pinned environment (Python 3.12)**: open the starter, Copy & Edit, File → Import Notebook with this file, keep "Pin to original environment" (LEARNINGS 2026-10-31). The first code cell stops on any other Python. |
| Attach | competition `gemma-4-developer-agent` (dev graphs + embeddings fit the embedding map) · dataset `metric/gemma-4-developer-agent-wheelhouse` |
| Optional | Kaggle Secret `GITHUB_TOKEN` (raises the GitHub API limit; without it the notebook waits for the hourly reset when needed) |
| Expected run time | ≈ 2–4 h (build ≈ 35 min, validation ≈ 30 min, issue text ≤ 1 h without a token, F2P/annotation ≈ 1–1.5 h, embeddings ≈ 30 min) |
| Output to bring back | `/kaggle/working/D0_summary.json` (aggregates + our own task ids/verdicts; commit to `results/` on pilot/d-heldout). The export `/kaggle/working/heldout_export/` is attached by D1/D2 as this notebook's output. Do not commit it. |


In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), f"Python {sys.version.split()[0]}: use the starter's pinned environment (3.12)"
print(sys.version)

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
COMP = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WH = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
W = Path('/kaggle/working'); PD = W / 'pd'; H = W / 'pool'; OUT = W / 'heldout_export'
for d in (PD, H): d.mkdir(parents=True, exist_ok=True)
wheels = [str(w) for w in WH.glob('*.whl') if w.name.split('-')[0].lower() in ('swegemma', 'adk_eval_core', 'adk_submission', 'google_adk', 'google_genai')]
assert len(wheels) == 5, wheels
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *wheels], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'git+https://github.com/danielmantiglia/codegraph-loc@1e8e0abbb97a1f78648cb12890fd700712b00623'], check=True)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)
import swegemma; print('swegemma', swegemma.__file__)

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/__init__.py', 'w').write(''); print('wrote common/__init__.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/mining.py', 'w').write('"""Mine candidate bug-fix tasks from a local git clone (no GitHub API needed).\n\nEvery filter step is counted so yield can be reported. Test/config classification mirrors\nswegemma.harness.verification._is_protected_test_or_config_path, because that is what the\nscorer strips from agent patches and resets before applying test_patch.\n\nUsage:\n  python -I common/mining.py REPO_DIR --repo owner/name --pkg PKGDIR --since 2026-05-01 \\\n      --out CANDIDATES.jsonl --funnel FUNNEL.json [--max-code-files 3] [--max-code-lines 200]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport subprocess\nfrom pathlib import PurePosixPath\n\nPROTECTED_CONFIG = {"conftest.py", "pytest.ini", "pyproject.toml", "tox.ini", "setup.cfg", ".pytest.ini",\n                    "sitecustomize.py", "usercustomize.py", "_swegemma_stubs.py"}\nFIX_RE = re.compile(r"^(fix|bug|bugfix|hotfix)\\b|^BUG\\b|^fix\\(", re.I)\nISSUE_REF_RE = re.compile(r"\\b(?:fix(?:es|ed)?|close[sd]?|resolve[sd]?)\\s*:?\\s+#(\\d+)", re.I)\nTRAILING_PR_RE = re.compile(r"\\(#(\\d+)\\)\\s*$")\nDOUBLE_REF_RE = re.compile(r"\\(#(\\d+)\\)\\s*\\(#(\\d+)\\)\\s*$")  # sqlglot style: (#issue) (#PR)\nTEST_ISSUE_RE = re.compile(r"(?:\\bgh-|github\\.com/[\\w.-]+/[\\w.-]+/issues/|\\bissue\\s+#?)(\\d{2,6})", re.I)\nAI_RE = re.compile(r"\\[CODEX\\]|Co-authored-by:.*(Claude|Copilot|Codex|Devin|cursor)|Generated with \\[?Claude|"\n                   r"copilot-swe-agent|\\bAI-assisted\\b", re.I)\n\n\ndef is_test_path(p: str) -> bool:\n    name = PurePosixPath(p).name\n    parts = PurePosixPath(p).parts\n    if not p.endswith(".py"):\n        return any(d in ("tests", "test", "testing") for d in parts[:-1])  # fixtures/data under tests/\n    return (name.startswith("test_") or name.endswith("_test.py")\n            or any(d in ("tests", "test", "testing") for d in parts[:-1]))\n\n\ndef is_protected_config(p: str) -> bool:\n    name = PurePosixPath(p).name\n    return name in PROTECTED_CONFIG or name.endswith(".pth")\n\n\ndef git(repo: str, *args: str) -> str:\n    return subprocess.run(["git", "-C", repo, *args], capture_output=True, text=True, check=True).stdout\n\n\ndef to_dev_style(diff: str) -> str:\n    """Rewrite `git diff` output into the dev set\'s style: only \'--- a/X\' / \'+++ b/X\' headers,\n    \'--- a/X\' even for new files, \'+++ b/X\' even for deleted files, no trailing newline."""\n    out, cur_old, cur_new = [], None, None\n    for line in diff.splitlines():\n        if line.startswith("diff --git "):\n            m = re.match(r"diff --git a/(.+) b/(.+)$", line)\n            cur_old, cur_new = m.group(1), m.group(2)\n            continue\n        if line.startswith(("index ", "new file mode", "deleted file mode", "similarity index",\n                            "rename from", "rename to", "old mode", "new mode")):\n            continue\n        if line.startswith("--- "):\n            out.append(f"--- a/{cur_old}")\n            continue\n        if line.startswith("+++ "):\n            out.append(f"+++ b/{cur_new}")\n            continue\n        out.append(line)\n    return "\\n".join(out)\n\n\n_PY_ATTR = None\n\n\ndef _python_diff_attributes() -> str:\n    """A gitattributes file enabling git\'s built-in Python diff driver, so hunk headers name the enclosing\n    `def`/`class` (e.g. \'@@ ... @@ def test_sqlite(self):\') as in the dev tasks (527 of 720 dev test-patch hunk\n    headers are `def ...`). This matters: when a test patch adds no `def test_*`, the scorer takes the required\n    tests from these headers (verification._extract_test_functions_from_patch)."""\n    global _PY_ATTR\n    if _PY_ATTR is None:\n        import tempfile\n        fd, _PY_ATTR = tempfile.mkstemp(prefix="pyattr_")\n        with open(fd, "w") as fh:\n            fh.write("*.py diff=python\\n")\n    return _PY_ATTR\n\n\ndef file_diff(repo: str, parent: str, commit: str, paths: list[str]) -> str:\n    if not paths:\n        return ""\n    return subprocess.run(["git", "-c", f"core.attributesFile={_python_diff_attributes()}", "-C", repo, "diff",\n                           "--no-color", "--no-renames", "-U3", parent, commit, "--", *paths],\n                          capture_output=True, text=True, check=True).stdout\n\n\ndef mine(args) -> None:\n    repo = args.repo_dir\n    shas = git(repo, "log", "--first-parent", f"--since={args.since}", "--format=%H", args.branch).split()\n    funnel = {"commits_since": len(shas)}\n    steps = ["single_parent", "fix_message", "has_test_and_code_change", "has_py_test_file", "pkg_python_only",\n             "no_protected_config_in_code", "size_ok"]\n    for s in steps:\n        funnel[s] = 0\n    funnel["with_issue_ref_in_message"] = 0\n    funnel["ai_authored_marker"] = 0\n    out = open(args.out, "w")\n    short = args.repo.split("/")[1].lower()\n    for sha in shas:\n        parents = git(repo, "log", "-1", "--format=%P", sha).split()\n        if len(parents) != 1:\n            continue\n        funnel["single_parent"] += 1\n        subject = git(repo, "log", "-1", "--format=%s", sha).strip()\n        body = git(repo, "log", "-1", "--format=%b", sha)\n        if not FIX_RE.search(subject):\n            continue\n        funnel["fix_message"] += 1\n        parent = parents[0]\n        names = [l.split("\\t") for l in git(repo, "diff", "--no-renames", "--name-status", parent, sha).splitlines()]\n        files = [n[-1] for n in names]\n        test_files = [f for f in files if is_test_path(f)]\n        code_files = [f for f in files if not is_test_path(f)]\n        code_py = [f for f in code_files if f.endswith(".py")]\n        if not test_files or not code_py:\n            continue\n        funnel["has_test_and_code_change"] += 1\n        # The harness runs pytest on the .py files of test_patch; fixture-only test patches\n        # make it pass a non-.py path to pytest ("no match"), so require a .py test file.\n        if not any(f.endswith(".py") and PurePosixPath(f).name != "conftest.py" for f in test_files):\n            continue\n        funnel["has_py_test_file"] += 1\n        # Code change must be Python inside the package; allow docs/changelog alongside (dropped from gold).\n        pkg_files = [f for f in code_files if f.startswith(args.pkg + "/")]\n        if any(not f.endswith(".py") for f in pkg_files) or not all(f in pkg_files for f in code_py):\n            continue\n        funnel["pkg_python_only"] += 1\n        if any(is_protected_config(f) for f in code_files):\n            continue\n        funnel["no_protected_config_in_code"] += 1\n        code_diff = file_diff(repo, parent, sha, pkg_files)\n        added = sum(1 for l in code_diff.splitlines() if l.startswith(("+", "-")) and not l.startswith(("+++", "---")))\n        if len(pkg_files) > args.max_code_files or added > args.max_code_lines:\n            continue\n        funnel["size_ok"] += 1\n        m2 = DOUBLE_REF_RE.search(subject)\n        issue_refs = sorted({int(x) for x in ISSUE_REF_RE.findall(subject + "\\n" + body)}\n                            | ({int(m2.group(1))} if m2 else set()))\n        pr = int(m2.group(2)) if m2 else (int(TRAILING_PR_RE.search(subject).group(1)) if TRAILING_PR_RE.search(subject) else None)\n        issue_refs = [i for i in issue_refs if i != pr]\n        test_diff = file_diff(repo, parent, sha, test_files)\n        added_test_text = "\\n".join(l for l in test_diff.splitlines() if l.startswith("+"))\n        issue_refs_in_tests = sorted({int(x) for x in TEST_ISSUE_RE.findall(added_test_text)} - {pr})\n        funnel["with_issue_ref_in_tests"] = funnel.get("with_issue_ref_in_tests", 0) + bool(issue_refs_in_tests)\n        funnel["with_any_issue_ref"] = funnel.get("with_any_issue_ref", 0) + bool(issue_refs or issue_refs_in_tests)\n        funnel["with_issue_ref_in_message"] += bool(issue_refs)\n        ai = bool(AI_RE.search(subject + "\\n" + body))\n        funnel["ai_authored_marker"] += ai\n        rec = {\n            "instance_id": f"{short}_{pr or sha[:10]}",\n            "repo": args.repo,\n            "base_commit": parent,\n            "fix_commit": sha,\n            "pr_number": pr,\n            "issue_refs": issue_refs,\n            "issue_refs_in_tests": issue_refs_in_tests,\n            "subject": subject,\n            "committed_at": git(repo, "log", "-1", "--format=%cI", sha).strip(),\n            "authored_at": git(repo, "log", "-1", "--format=%aI", sha).strip(),\n            "code_files": pkg_files,\n            "test_files": test_files,\n            "code_lines_changed": added,\n            "ai_authored_marker": ai,\n            "patch": to_dev_style(code_diff),\n            "test_patch": to_dev_style(test_diff),\n        }\n        out.write(json.dumps(rec) + "\\n")\n    json.dump(funnel, open(args.funnel, "w"), indent=1)\n    print(json.dumps(funnel))\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("repo_dir")\n    p.add_argument("--repo", required=True)\n    p.add_argument("--pkg", required=True)\n    p.add_argument("--since", default="2026-05-01")\n    p.add_argument("--branch", default="HEAD")\n    p.add_argument("--out", required=True)\n    p.add_argument("--funnel", required=True)\n    p.add_argument("--max-code-files", type=int, default=3)\n    p.add_argument("--max-code-lines", type=int, default=200)\n    mine(p.parse_args())\n'); print('wrote common/mining.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/snapshot.py', 'w').write('"""Build a task snapshot .tgz in the same layout as the competition\'s snapshots/<instance_id>.tgz.\n\nObserved dev layout (schema only): archive rooted at \'./\', a git repo whose history is\n`git fast-export` of the source up to base_commit piped into `git fast-import` (so commit hashes\ndiffer from upstream; tasks.jsonl keeps the upstream base_commit), branches `export_ref` and\n`main` (checked out) at the base, .git/config with user agent@eval / Agent, clean working tree.\nHistory depth is whatever the source clone has (dev snapshots range from 4 to 6k commits).\n\nUsage: python -I common/snapshot.py REPO_DIR BASE_COMMIT OUT.tgz\n"""\nfrom __future__ import annotations\n\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\n\ndef run(*cmd: str, cwd: str | None = None, stdin=None, stdout=None) -> None:\n    subprocess.run(cmd, cwd=cwd, check=True, stdin=stdin, stdout=stdout, stderr=subprocess.PIPE)\n\n\ndef build_snapshot(repo_dir: str, base_commit: str, out_tgz: str) -> dict:\n    out = Path(out_tgz)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory() as td:\n        dst = Path(td) / "ws"\n        dst.mkdir()\n        run("git", "init", "-q", "-b", "main", str(dst))\n        run("git", "-C", repo_dir, "branch", "-f", "export_ref", base_commit)\n        exp = subprocess.Popen(["git", "-C", repo_dir, "fast-export", "--signed-tags=strip",\n                                "--tag-of-filtered-object=drop", "--reencode=yes", "refs/heads/export_ref"],\n                               stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)\n        run("git", "-C", str(dst), "fast-import", "--quiet", stdin=exp.stdout)\n        if exp.wait() != 0:\n            raise RuntimeError("git fast-export failed")\n        run("git", "-C", str(dst), "checkout", "-q", "-B", "main", "export_ref")\n        run("git", "-C", str(dst), "config", "user.email", "agent@eval")\n        run("git", "-C", str(dst), "config", "user.name", "Agent")\n        run("git", "-C", str(dst), "gc", "-q", "--prune=now")\n        # Same top-level shape as dev archives: entries prefixed with \'./\'\n        run("tar", "-C", str(dst), "-czf", str(out), ".")\n        n_commits = int(subprocess.run(["git", "-C", str(dst), "rev-list", "--count", "HEAD"],\n                                       capture_output=True, text=True).stdout.strip() or 0)\n    return {"snapshot_bytes": out.stat().st_size, "history_commits": n_commits}\n\n\nif __name__ == "__main__":\n    print(build_snapshot(sys.argv[1], sys.argv[2], sys.argv[3]))\n'); print('wrote common/snapshot.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/validate.py', 'w').write('"""Validate tasks with the competition harness\'s own Phase-2 code (swegemma.verify_task).\n\nA task is VALID iff, in the chosen sandbox backend:\n  baseline: verify_task(agent_patch="")          -> resolved == False   (tests fail without the fix)\n  gold:     verify_task(agent_patch=task.patch)  -> resolved == True    (exit 0 + JUnit checks pass)\nThis is the host\'s own criterion (swegemma.harness.sample_verification: base_exit != 0 and\npatch_exit == 0), but run through verify_task so protected-file stripping, test-file reset,\npytest.ini/conftest rewriting and JUnit validation are exactly what the scorer does.\n\nRun with the interpreter that has swegemma installed. In subprocess mode that interpreter\'s\nsite-packages ARE the sandbox\'s environment (SubprocessManager uses system_site_packages=True),\nso it should mirror the Kaggle notebook host as closely as possible.\n\nUsage:\n  python common/validate.py --tasks T.jsonl --snapshots DIR --backend subprocess \\\n      --out results.jsonl [--wheels DIR] [--ids a b c] [--logs DIR]\nOutput: one JSON line per task with verdict fields only (no patches, no test output).\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport asyncio\nimport json\nimport logging\nimport sys\nimport tempfile\nimport time\nfrom pathlib import Path\n\n\ndef _config(args, results_dir: Path):\n    from adk_submission import ModelRegistry\n    from swegemma.config import EvalConfig\n\n    return EvalConfig(\n        tasks_path=Path(args.tasks),\n        snapshots_dir=Path(args.snapshots),\n        results_dir=results_dir,\n        submission_dir=results_dir,\n        models=ModelRegistry(),\n        sandbox=args.backend,\n        image=args.image,\n        wheels_dir=Path(args.wheels) if args.wheels else None,\n        timeout_seconds=args.timeout,\n        verification_timeout_seconds=args.timeout,\n    )\n\n\ndef _sandbox(config):\n    if config.sandbox == "docker":\n        from swegemma.sandbox import ContainerConfig, ContainerManager\n\n        return ContainerManager(ContainerConfig(image=config.image, timeout_seconds=config.verification_timeout_seconds))\n    from swegemma.sandbox.subprocess import SubprocessManager\n\n    return SubprocessManager(timeout_seconds=config.verification_timeout_seconds)\n\n\ndef _first_error_line(text: str) -> str:\n    """A short, data-free failure category (exception class / pytest summary keyword)."""\n    import re\n\n    for pat in (r"(ModuleNotFoundError|ImportError|AttributeError|TypeError|NameError|SyntaxError|"\n                r"AssertionError|FileNotFoundError|fixture \'[^\']+\' not found|ERROR collecting)",\n                r"(\\d+ failed|\\d+ error)"):\n        m = re.search(pat, text or "")\n        if m:\n            return m.group(1)[:80]\n    return ""\n\n\nasync def _one(task, args, logs: Path | None):\n    from swegemma.harness.verification import verify_task\n\n    snap = Path(args.snapshots) / f"{task.instance_id}.tgz"\n    if not snap.exists():\n        snap = Path(args.snapshots) / f"{task.instance_id}.tar.gz"\n    rec = {"instance_id": task.instance_id, "repo": task.repo, "backend": args.backend}\n    with tempfile.TemporaryDirectory() as td:\n        config = _config(args, Path(td))\n        sb = _sandbox(config)\n        for kind, patch in (("base", ""), ("gold", task.patch)):\n            t0 = time.perf_counter()\n            try:\n                r = await verify_task(sb, config, task, snap, agent_patch=patch, start_time=t0)\n                rec[f"{kind}_resolved"] = bool(r.resolved)\n                rec[f"{kind}_exit"] = r.test_exit_code\n                rec[f"{kind}_error"] = (r.error or "")[:160]\n                rec[f"{kind}_symptom"] = _first_error_line(r.test_output)\n                if logs:\n                    (logs / f"{task.instance_id}.{args.backend}.{kind}.log").write_text(\n                        f"error: {r.error}\\n\\n{r.test_output}")\n            except Exception as e:  # harness/setup failure, not a test verdict\n                rec[f"{kind}_resolved"] = None\n                rec[f"{kind}_error"] = f"{type(e).__name__}: {e}"[:160]\n            rec[f"{kind}_seconds"] = round(time.perf_counter() - t0, 1)\n    rec["valid"] = rec.get("base_resolved") is False and rec.get("gold_resolved") is True\n    return rec\n\n\nasync def main_async(args):\n    from swegemma.models import load_tasks\n\n    tasks = load_tasks(Path(args.tasks))\n    if args.ids:\n        tasks = [t for t in tasks if t.instance_id in set(args.ids)]\n    logs = Path(args.logs) if args.logs else None\n    if logs:\n        logs.mkdir(parents=True, exist_ok=True)\n    out = open(args.out, "a")\n    sem = asyncio.Semaphore(args.concurrency)\n\n    async def run(t):\n        async with sem:\n            rec = await _one(t, args, logs)\n            out.write(json.dumps(rec) + "\\n")\n            out.flush()\n            print(json.dumps({k: rec.get(k) for k in ("instance_id", "valid", "base_resolved", "gold_resolved",\n                                                      "gold_symptom", "gold_error", "gold_seconds")}), flush=True)\n\n    await asyncio.gather(*(run(t) for t in tasks))\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--tasks", required=True)\n    p.add_argument("--snapshots", required=True)\n    p.add_argument("--backend", choices=["subprocess", "docker"], default="subprocess")\n    p.add_argument("--image", default="swebench-sandbox:latest")\n    p.add_argument("--wheels")\n    p.add_argument("--ids", nargs="*")\n    p.add_argument("--out", required=True)\n    p.add_argument("--logs")\n    p.add_argument("--timeout", type=int, default=300)\n    p.add_argument("--concurrency", type=int, default=2)\n    a = p.parse_args()\n    logging.basicConfig(level=logging.WARNING)\n    asyncio.run(main_async(a))\n'); print('wrote common/validate.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/testmap.py', 'w').write('"""Compute FAIL_TO_PASS / PASS_TO_PASS with the harness\'s own sandbox setup.\n\nFor each task: fresh sandbox -> extract snapshot -> harness test config -> baseline commit ->\napply test_patch -> pytest (JUnit) [state "base"]; then apply the gold patch -> pytest [state "gold"].\nF2P = tests not passing at base and passing at gold; P2P = passing in both.\nNode ids use the SWE-bench convention `path::Class::test[param]`.\n\nUsage: python common/testmap.py --tasks T.jsonl --snapshots DIR --out TESTMAP.jsonl [--ids ...]\n(run with the interpreter that has swegemma installed; subprocess backend)\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport tempfile\nimport xml.etree.ElementTree as ET\nfrom pathlib import Path\n\nfrom swegemma.harness.container_setup import (apply_patch_in_container, extract_snapshot, setup_baseline_commit,\n                                              setup_git_exclude, setup_workspace_test_config)\nfrom swegemma.models import load_tasks\nfrom swegemma.sandbox.subprocess import SubprocessManager\n\n\n# Per-test timeout (pytest-timeout) used ONLY for labelling: a hang bug otherwise kills the whole\n# base run, no JUnit is written, and every test of the file would be mislabelled FAIL_TO_PASS.\n# Validity verdicts still come from verify_task, which runs with the harness\'s own timeout=0.\nPER_TEST_TIMEOUT = 60\n\n\ndef _outcomes(sb, sid, targets: list[str]) -> dict[str, str]:\n    xml = "/tmp/_testmap_junit.xml"\n    cmd = (f"cd /workspace && rm -f {xml} && PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 python3 -s -m pytest "\n           f"{\' \'.join(targets)} --junitxml={xml} -o junit_family=xunit1 -p no:anyio -o timeout={PER_TEST_TIMEOUT} "\n           f"-o python_classes=\'Test* *Test\' -q >/dev/null 2>&1; cat {xml}")\n    out = sb.exec(sid, cmd).stdout or ""\n    res: dict[str, str] = {}\n    try:\n        root = ET.fromstring(out[out.find("<"):])\n    except ET.ParseError:\n        return res\n    for tc in root.iter("testcase"):\n        f, cls, name = tc.get("file", ""), tc.get("classname", ""), tc.get("name", "")\n        mod = f[:-3].replace("/", ".") if f.endswith(".py") else ""\n        inner = cls[len(mod) + 1:] if mod and cls.startswith(mod + ".") else ""\n        node = "::".join(x for x in (f, inner.replace(".", "::"), name) if x)\n        state = ("failed" if tc.find("failure") is not None else "error" if tc.find("error") is not None\n                 else "skipped" if tc.find("skipped") is not None else "passed")\n        res[node] = state\n    return res\n\n\ndef _apply(sb, sid, patch: str) -> int:\n    with tempfile.NamedTemporaryFile("w", suffix=".patch", delete=False) as fh:\n        fh.write(patch if patch.endswith("\\n") else patch + "\\n")\n    sb.copy_to(sid, Path(fh.name), "/tmp/")\n    code, _, _ = apply_patch_in_container(sb, sid, f"/tmp/{Path(fh.name).name}")\n    Path(fh.name).unlink(missing_ok=True)\n    return code\n\n\ndef testmap(task, snap: Path) -> dict:\n    sb = SubprocessManager(timeout_seconds=600)\n    sid = sb.start()\n    try:\n        extract_snapshot(sb, sid, snap)\n        setup_git_exclude(sb, sid)\n        setup_workspace_test_config(sb, sid, repo=task.repo)\n        setup_baseline_commit(sb, sid, "eval_baseline")\n        targets = [p for p in task.test_files if p.endswith(".py") and not p.endswith("conftest.py")]\n        if _apply(sb, sid, task.test_patch) != 0:\n            return {"instance_id": task.instance_id, "error": "test_patch did not apply"}\n        base = _outcomes(sb, sid, targets)\n        if _apply(sb, sid, task.patch) != 0:\n            return {"instance_id": task.instance_id, "error": "gold patch did not apply"}\n        gold = _outcomes(sb, sid, targets)\n    finally:\n        sb.stop(sid)\n    f2p = sorted(n for n, s in gold.items() if s == "passed" and base.get(n) != "passed")\n    p2p = sorted(n for n, s in gold.items() if s == "passed" and base.get(n) == "passed")\n    broken = sorted(n for n, s in gold.items() if s in ("failed", "error"))\n    skipped = sorted(n for n, s in gold.items() if s == "skipped")\n    return {"instance_id": task.instance_id, "FAIL_TO_PASS": f2p, "PASS_TO_PASS": p2p,\n            "gold_not_passing": broken, "gold_skipped": skipped, "n_tests": len(gold)}\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--tasks", required=True)\n    p.add_argument("--snapshots", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--ids", nargs="*")\n    a = p.parse_args()\n    with open(a.out, "w") as fh:\n        for t in load_tasks(Path(a.tasks)):\n            if a.ids and t.instance_id not in a.ids:\n                continue\n            rec = testmap(t, Path(a.snapshots) / f"{t.instance_id}.tgz")\n            fh.write(json.dumps(rec) + "\\n")\n            print(t.instance_id, {k: (len(v) if isinstance(v, list) else v) for k, v in rec.items() if k != "instance_id"},\n                  flush=True)\n'); print('wrote common/testmap.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/annotate.py', 'w').write('"""Intervention-ready annotations for swegemma-format tasks (held-out or dev).\n\nPer task:\n  edit_hunks:   each code-patch hunk -> file, old line span, enclosing symbol as a graph node id\n                (released id rule, common/graph_build.released_ids_for_file) or "<module>" if at module level\n  gold_nodes:   unique node ids edited by the gold patch (localization ground truth, graph id space)\n  gold_nodes_missing_from_graph_rule: edited symbols the released-style graph cannot contain\n                (module-level code, async defs)\n  added_symbols: defs/classes that only exist after the patch (new functions)\n  repro_test:   (needs FAIL_TO_PASS) a minimal runnable pytest file: test-file imports and module-level\n                statements + only the F2P tests (TestCase classes keep setUp/fixtures/helpers), with\n                --verify: run in the harness sandbox, must FAIL before and PASS after the gold patch.\n\nUsage (interpreter with swegemma + codegraph-loc installed):\n  python common/annotate.py --tasks T.jsonl --snapshots DIR --out ANN.jsonl [--testmap TM.jsonl] [--verify]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport io\nimport json\nimport re\nimport sys\nimport tarfile\nimport tempfile\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom common.graph_build import released_ids_for_file  # noqa: E402\n\nHUNK_RE = re.compile(r"^@@ -(\\d+)(?:,(\\d+))? \\+(\\d+)(?:,(\\d+))? @@")\n\n\ndef parse_hunks(patch: str):\n    """Yield (file, old_start, old_len, new_start, new_len, changed_old_lines, changed_new_lines)."""\n    cur, h = None, None\n    for line in patch.splitlines():\n        if line.startswith("+++ b/"):\n            cur = line[6:]\n            continue\n        if line.startswith("--- "):\n            continue\n        m = HUNK_RE.match(line)\n        if m:\n            if h:\n                yield h\n            os_, ol, ns, nl = int(m[1]), int(m[2] or 1), int(m[3]), int(m[4] or 1)\n            h = {"file": cur, "old_start": os_, "old_len": ol, "new_start": ns, "new_len": nl,\n                 "old_changed": [], "new_changed": [], "_o": os_, "_n": ns}\n            continue\n        if h is None:\n            continue\n        if line.startswith("-"):\n            h["old_changed"].append(h["_o"]); h["_o"] += 1\n        elif line.startswith("+"):\n            h["new_changed"].append(h["_n"]); h["_n"] += 1\n        else:\n            h["_o"] += 1; h["_n"] += 1\n    if h:\n        yield h\n\n\ndef _defs_with_spans(src: str):\n    try:\n        tree = ast.parse(src)\n    except SyntaxError:\n        return []\n    out = []\n    for n in ast.walk(tree):\n        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):\n            start = min([n.lineno] + [d.lineno for d in n.decorator_list])\n            out.append((start, n.end_lineno, n.lineno, isinstance(n, ast.AsyncFunctionDef)))\n    return out\n\n\ndef enclosing(defs, line):\n    best = None\n    for s, e, defline, is_async in defs:\n        if s <= line <= e and (best is None or s >= best[0]):\n            best = (s, e, defline, is_async)\n    return best\n\n\ndef module_name(root: Path, rel: str) -> str:\n    from cgl.graph import module_name_for\n    return module_name_for(root / rel, root, scheme="official")[0]\n\n\ndef locate(root: Path, task: dict) -> dict:\n    hunks, gold, missing = [], set(), set()\n    new_files = {}\n    for h in parse_hunks(task["patch"]):\n        rel = h["file"]\n        if not rel.endswith(".py"):\n            continue\n        f = root / rel\n        mod = module_name(root, rel)\n        src = f.read_text(encoding="utf-8", errors="replace") if f.exists() else ""\n        defs, ids = _defs_with_spans(src), released_ids_for_file(f, mod) if f.exists() else {}\n        lines = h["old_changed"] or [max(1, h["old_start"] + (h["old_len"] - 1) // 2)]  # pure insertion: anchor\n        syms = set()\n        for ln in lines:\n            enc = enclosing(defs, ln)\n            if enc is None:\n                syms.add(f"{mod}.<module>"); missing.add(f"{mod}.<module>")\n            elif enc[2] in ids:\n                syms.add(ids[enc[2]])\n            else:  # async def or other def the released graphs do not contain\n                tag = f"{mod}.<async@{enc[2]}>" if enc[3] else f"{mod}.<def@{enc[2]}>"\n                syms.add(tag); missing.add(tag)\n        gold |= {s for s in syms if "<" not in s}\n        hunks.append({"file": rel, "old_start": h["old_start"], "old_len": h["old_len"], "symbols": sorted(syms)})\n        new_files.setdefault(rel, []).extend(h["new_changed"])\n    return {"edit_hunks": hunks, "gold_nodes": sorted(gold),\n            "gold_nodes_missing_from_graph_rule": sorted(missing), "_new_lines": new_files}\n\n\ndef added_symbols(new_root: Path, new_lines: dict, old_ids: set[str]) -> list[str]:\n    out = set()\n    for rel, lines in new_lines.items():\n        f = new_root / rel\n        if not f.exists():\n            continue\n        ids = released_ids_for_file(f, module_name(new_root, rel))\n        defs = _defs_with_spans(f.read_text(encoding="utf-8", errors="replace"))\n        for ln in lines:\n            enc = enclosing(defs, ln)\n            if enc and enc[2] in ids and ids[enc[2]] not in old_ids:\n                out.add(ids[enc[2]])\n    return sorted(out)\n\n\n# ---------------- reproduction tests ----------------\n\ndef build_repro(test_src: str, f2p_nodes: list[str], test_path: str) -> str | None:\n    """Keep imports + module-level non-test statements + only the selected tests."""\n    tree = ast.parse(test_src)\n    want_funcs, want_methods = set(), {}\n    for node in f2p_nodes:\n        parts = node.split("::")\n        if parts[0] != test_path:\n            continue\n        name = parts[-1].split("[", 1)[0]\n        if len(parts) == 2:\n            want_funcs.add(name)\n        else:\n            want_methods.setdefault(parts[1], set()).add(name)\n    if not want_funcs and not want_methods:\n        return None\n    body = []\n    for st in tree.body:\n        if isinstance(st, (ast.FunctionDef, ast.AsyncFunctionDef)):\n            if st.name.startswith("test"):\n                if st.name in want_funcs:\n                    body.append(st)\n            else:\n                body.append(st)  # helpers and fixtures\n        elif isinstance(st, ast.ClassDef):\n            if st.name in want_methods:\n                keep = [b for b in st.body if not (isinstance(b, (ast.FunctionDef, ast.AsyncFunctionDef))\n                                                   and b.name.startswith("test") and b.name not in want_methods[st.name])]\n                st.body = keep or [ast.Pass()]\n                body.append(st)\n            elif not st.name.startswith("Test"):\n                body.append(st)  # helper / base classes\n        else:\n            body.append(st)\n    tree.body = body\n    return ast.unparse(tree) + "\\n"\n\n\ndef _sandbox_run_repro(snap: Path, repo: str, repro: str, gold_patch: str, test_patch: str):\n    from swegemma.harness.container_setup import (apply_patch_in_container, extract_snapshot, setup_baseline_commit,\n                                                  setup_git_exclude, setup_workspace_test_config)\n    from swegemma.sandbox.subprocess import SubprocessManager\n\n    def apply(sb, sid, patch):\n        with tempfile.NamedTemporaryFile("w", suffix=".patch", delete=False) as fh:\n            fh.write(patch if patch.endswith("\\n") else patch + "\\n")\n        sb.copy_to(sid, Path(fh.name), "/tmp/")\n        rc = apply_patch_in_container(sb, sid, f"/tmp/{Path(fh.name).name}")[0]\n        Path(fh.name).unlink(missing_ok=True)\n        return rc\n\n    sb = SubprocessManager(timeout_seconds=600)\n    sid = sb.start()\n    try:\n        extract_snapshot(sb, sid, snap)\n        setup_git_exclude(sb, sid)\n        setup_workspace_test_config(sb, sid, repo=repo)\n        setup_baseline_commit(sb, sid, "eval_baseline")\n        # test_patch may add fixtures/data the repro needs (it never adds library code)\n        apply(sb, sid, test_patch)\n        sb.write_file("/tmp/test_repro_injected.py", repro)\n        cmd = ("cd /workspace && PYTHONSAFEPATH=1 PYTHONNOUSERSITE=1 timeout 240 python3 -s -m pytest "\n               "/tmp/test_repro_injected.py -p no:anyio -p no:cacheprovider -o timeout=60 -q >/dev/null 2>&1; echo $?")\n        before = (sb.exec(sid, cmd).stdout or "").strip().splitlines()[-1:]\n        apply(sb, sid, gold_patch)\n        after = (sb.exec(sid, cmd).stdout or "").strip().splitlines()[-1:]\n    finally:\n        sb.stop(sid)\n    return (before[0] if before else "?"), (after[0] if after else "?")\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--tasks", required=True)\n    p.add_argument("--snapshots", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--testmap")\n    p.add_argument("--verify", action="store_true")\n    p.add_argument("--ids", nargs="*")\n    a = p.parse_args()\n    tm = {r["instance_id"]: r for r in map(json.loads, open(a.testmap))} if a.testmap else {}\n    with open(a.out, "w") as out:\n        for line in open(a.tasks):\n            t = json.loads(line)\n            if a.ids and t["instance_id"] not in a.ids:\n                continue\n            snap = Path(a.snapshots) / f"{t[\'instance_id\']}.tgz"\n            with tempfile.TemporaryDirectory() as td:\n                root = Path(td) / "ws"\n                with tarfile.open(snap) as tf:\n                    tf.extractall(root, filter="data")\n                rec = {"instance_id": t["instance_id"], **locate(root, t)}\n                # apply gold + test patch in a scratch copy to find added symbols and the post-test file\n                import subprocess\n                for patch in (t["patch"], t["test_patch"]):\n                    pf = Path(td) / "p.patch"\n                    pf.write_text(patch + ("\\n" if not patch.endswith("\\n") else ""))\n                    subprocess.run(["git", "-c", "safe.directory=*", "-C", str(root), "apply", "--unsafe-paths",\n                                    str(pf)], capture_output=True)\n                rec["added_symbols"] = added_symbols(root, rec.pop("_new_lines"), set(rec["gold_nodes"]))\n                f2p = tm.get(t["instance_id"], {}).get("FAIL_TO_PASS", [])\n                if f2p:\n                    test_path = f2p[0].split("::")[0]\n                    src = (root / test_path).read_text(encoding="utf-8", errors="replace")\n                    repro = build_repro(src, f2p, test_path)\n                    rec["repro_test"] = {"source_file": test_path, "n_f2p": len(f2p), "code": repro}\n                    if repro and a.verify:\n                        rc_before, rc_after = _sandbox_run_repro(snap, t["repo"], repro, t["patch"], t["test_patch"])\n                        rec["repro_test"].update(exit_before_gold=rc_before, exit_after_gold=rc_after,\n                                                 verified=(rc_before not in ("0", "?") and rc_after == "0"))\n            out.write(json.dumps(rec) + "\\n")\n            print(t["instance_id"], len(rec["gold_nodes"]), "gold nodes;",\n                  "missing", len(rec["gold_nodes_missing_from_graph_rule"]),\n                  "| repro verified:", rec.get("repro_test", {}).get("verified"), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote common/annotate.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/graph_build.py', 'w').write('"""Code graphs in the competition\'s released node-link schema, built with codegraph-loc.\n\nGenerator: codegraph-loc (Apache-2.0, https://github.com/danielmantiglia/codegraph-loc,\ncommit 1e8e0ab, D. Mantiglia 2026), used as a library: `cgl.graph.build_graph` +\n`cgl.graph.official_view` (released id convention). This module adds a "released view"\nreverse-engineered from the released graphs (schema notes in docs/FORMAT.md):\n  * nodes: classes, functions, methods only (no module nodes); async defs dropped unless --fixed\n  * node text: functions from the `def` keyword (original indentation kept); classes rendered as\n    a skeleton: header, cleaned docstring (or `None`), then method signatures with `...` bodies\n  * edges: type "calls", key 0, and SYMMETRIC: every containment pair and every resolved call is\n    stored in both directions (in the released graphs 100% of edges have their reverse)\n  * graph attribute repo_name = "<repo_short>_<base_commit>"; file name graphs/<repo_short>_<base_commit>.json\n\nUsage:\n  python common/graph_build.py build REPO_ROOT REPO_SHORT BASE_COMMIT OUT_DIR [--fixed]\n  python common/graph_build.py fidelity OURS.json RELEASED.json\n"""\nfrom __future__ import annotations\n\nimport ast\nimport json\nimport sys\nfrom pathlib import Path\n\n\ndef _func_sig(n) -> str:\n    s = f"def {n.name}({ast.unparse(n.args)})"\n    if n.returns is not None:\n        s += f" -> {ast.unparse(n.returns)}"\n    return s + ":"\n\n\ndef _class_text(node: ast.ClassDef) -> str:\n    bases = [ast.unparse(b) for b in node.bases] + [ast.unparse(k) for k in node.keywords]\n    head = f"class {node.name}" + (f"({\', \'.join(bases)})" if bases else "") + ":"\n    doc = ast.get_docstring(node)\n    parts = [head, f\'    """{doc}"""\' if doc is not None else "    None", ""]\n    for b in node.body:\n        if isinstance(b, (ast.FunctionDef, ast.AsyncFunctionDef)):\n            for d in b.decorator_list:\n                parts.append(f"    @{ast.unparse(d)}")\n            parts.append(f"    {_func_sig(b)}\\n        ...\\n")\n    return "\\n".join(parts).rstrip()\n\n\ndef _texts_for_file(path: Path) -> dict[int, str]:\n    """Map definition line (the `def`/`class` keyword line) -> released-style text."""\n    src = path.read_text(encoding="utf-8", errors="replace")\n    try:\n        tree = ast.parse(src)\n    except SyntaxError:\n        return {}\n    out = {}\n    for n in ast.walk(tree):\n        if isinstance(n, ast.ClassDef):\n            out[n.lineno] = _class_text(n)\n        elif isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)):\n            seg = ast.get_source_segment(src, n) or ""\n            out[n.lineno] = seg[seg.find("def ") if not seg.startswith("def ") else 0:] if "def " in seg else seg\n    return out\n\n\nRELEASED_CALL_RESOLUTIONS = {"module", "import", "local"}  # no self./typed/unique-name/super in released graphs\n\n\ndef released_ids_for_file(path: Path, module: str, fixed: bool = False) -> dict[int, str]:\n    """Exact released id rule (reproduces 100% of node ids on the released requests/httpx graphs).\n\n    The released generator keeps ONE "current class" variable, set on entering a class body and\n    reset to None (not restored) on leaving it. So: classes -> <module>.<Class>; any def while a\n    class is current -> <module>.<Class>.<name> (closures inside methods included); any def after a\n    nested class body has closed inside an outer class -> <module>.<name> (the documented\n    "methods labelled as module-level" bug). Async defs are skipped (released graphs have none).\n    With fixed=True a proper class stack is used and async defs are kept.\n    Returns {line of def/class keyword (and of each decorator): id}."""\n    sys.setrecursionlimit(max(sys.getrecursionlimit(), 50000))  # deeply nested generated code (see build)\n    try:\n        tree = ast.parse(path.read_text(encoding="utf-8", errors="replace"))\n    except SyntaxError:\n        return {}\n    out: dict[int, str] = {}\n    state = {"cls": None}\n    func_types = (ast.FunctionDef, ast.AsyncFunctionDef) if fixed else (ast.FunctionDef,)\n\n    def mark(node, nid):\n        out[node.lineno] = nid\n        for d in node.decorator_list:\n            out.setdefault(d.lineno, nid)\n\n    def visit(node):\n        for ch in ast.iter_child_nodes(node):\n            if isinstance(ch, ast.ClassDef):\n                mark(ch, f"{module}.{ch.name}")\n                prev = state["cls"]\n                state["cls"] = ch.name\n                visit(ch)\n                state["cls"] = prev if fixed else None\n            elif isinstance(ch, func_types):\n                mark(ch, f"{module}.{state[\'cls\']}.{ch.name}" if state["cls"] else f"{module}.{ch.name}")\n                visit(ch)\n            else:\n                visit(ch)\n\n    visit(tree)\n    return out\n\n\ndef build(root: str, repo_short: str, base_commit: str, out_dir: str, fixed: bool = False) -> Path:\n    """Node ids/texts from our AST rules; call resolution from codegraph-loc, re-keyed by (file, line)."""\n    from cgl.graph import build_graph\n\n    # Deeply nested generated code (e.g. sympy) exceeds the default recursion limit in the AST visitors.\n    # Python >= 3.11 runs pure-Python recursion without consuming C stack, so a higher limit is safe.\n    sys.setrecursionlimit(max(sys.getrecursionlimit(), 50000))\n    G0 = build_graph(root, module_scheme="official")  # package-relative names inside packages, else repo path\n    rootp = Path(root)\n    ids_cache: dict[str, dict[int, str]] = {}\n    texts_cache: dict[str, dict[int, str]] = {}\n    mapping, keep = {}, {}\n    for n, a in G0.nodes(data=True):\n        if a.get("kind") not in ("class", "function", "method") or not a.get("file"):\n            continue\n        f = a["file"]\n        if f not in ids_cache:\n            ids_cache[f] = released_ids_for_file(rootp / f, a.get("module"), fixed)\n            texts_cache[f] = _texts_for_file(rootp / f)\n        sl = a.get("start_line") or 0\n        rid = ids_cache[f].get(sl)\n        if rid is None:\n            continue  # e.g. async def in faithful mode\n        mapping[n] = rid\n        if rid not in keep:  # released graphs collapse same-id defs into one node (first wins)\n            text = next((texts_cache[f][ln] for ln in range(sl, sl + 50) if ln in texts_cache[f]), a.get("text", ""))\n            keep[rid] = {"id": rid, "name": rid, "text": text}\n    kind = {n: a.get("kind") for n, a in G0.nodes(data=True)}\n    pairs = set()\n    for u, v, a in G0.edges(data=True):\n        if u not in mapping or v not in mapping:\n            continue\n        t = a.get("type")\n        ok = (t == "calls" and (fixed or a.get("resolution") in RELEASED_CALL_RESOLUTIONS)) or \\\n             (t == "contains" and (fixed or (kind.get(u) == "class" and kind.get(v) == "method")))\n        mu, mv = mapping[u], mapping[v]\n        if ok and mu != mv:\n            pairs.add((mu, mv))\n            pairs.add((mv, mu))\n    data = {"directed": True, "multigraph": True, "graph": {"repo_name": f"{repo_short}_{base_commit}"},\n            "nodes": list(keep.values()),\n            "edges": [{"type": "calls", "source": u, "target": v, "key": 0} for u, v in sorted(pairs)]}\n    out = Path(out_dir) / f"{repo_short}_{base_commit}.json"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(data))\n    return out\n\n\ndef fidelity(ours_path: str, released_path: str) -> dict:\n    o, r = json.load(open(ours_path)), json.load(open(released_path))\n    on, rn = {n["id"]: n for n in o["nodes"]}, {n["id"]: n for n in r["nodes"]}\n    oe = {frozenset((e["source"], e["target"])) for e in o["edges"]}\n    re_ = {frozenset((e["source"], e["target"])) for e in r["edges"]}\n    common = set(on) & set(rn)\n    re_in = {e for e in re_ if e <= set(on)}  # released edges whose endpoints we also have\n    return {\n        "released_nodes": len(rn), "our_nodes": len(on),\n        "node_recall": round(len(common) / max(1, len(rn)), 4),\n        "node_precision": round(len(common) / max(1, len(on)), 4),\n        "text_exact_on_common": round(sum(on[i]["text"] == rn[i]["text"] for i in common) / max(1, len(common)), 4),\n        "released_undirected_edges": len(re_), "our_undirected_edges": len(oe),\n        "edge_recall": round(len(oe & re_) / max(1, len(re_)), 4),\n        "edge_recall_on_shared_nodes": round(len(oe & re_in) / max(1, len(re_in)), 4),\n        "edge_precision": round(len(oe & re_) / max(1, len(oe)), 4),\n        "released_edges_all_reciprocal": all((e["target"], e["source"]) in {(x["source"], x["target"]) for x in r["edges"]}\n                                             for e in r["edges"]),\n    }\n\n\nif __name__ == "__main__":\n    if sys.argv[1] == "build":\n        print(build(sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5], fixed="--fixed" in sys.argv))\n    else:\n        print(json.dumps(fidelity(sys.argv[2], sys.argv[3])))\n'); print('wrote common/graph_build.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/github_text.py', 'w').write('"""Fill problem_statement / created_at from GitHub (issue first, PR body as fallback).\n\nRun where api.github.com is reachable (Kaggle notebook with Internet ON, or a laptop). Optional\nGITHUB_TOKEN (env var or Kaggle Secret) raises the limit to 5,000 req/h; unauthenticated is 60 req/h (about\n25-40 tasks per hour; the script waits for the hourly reset, bounded by GITHUB_MAX_RESET_WAIT).\n\nproblem_statement follows the SWE-bench convention: "<title>\\n<body>". created_at is the issue\'s\ncreation time (contamination dating). Provenance is recorded in meta: issue | pr_body.\n\nUsage: python common/github_text.py TASKS.jsonl META.jsonl OUT_TASKS.jsonl OUT_META.jsonl\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport sys\nimport time\nimport urllib.error\nimport urllib.request\n\n\ndef _token() -> str | None:\n    """GITHUB_TOKEN env var, else an optional Kaggle Secret of the same name (never required)."""\n    if os.environ.get("GITHUB_TOKEN"):\n        return os.environ["GITHUB_TOKEN"]\n    try:\n        from kaggle_secrets import UserSecretsClient  # only exists inside Kaggle notebooks\n        return UserSecretsClient().get_secret("GITHUB_TOKEN") or None\n    except Exception:\n        return None\n\n\nTOKEN = _token()\nMAX_RESET_WAIT = int(os.environ.get("GITHUB_MAX_RESET_WAIT", "3700"))  # seconds; unauthenticated limit resets hourly\n\n\ndef gh(path: str) -> dict | None:\n    req = urllib.request.Request(f"https://api.github.com{path}",\n                                 headers={"Accept": "application/vnd.github+json", "User-Agent": "heldout-builder"})\n    if TOKEN:\n        req.add_header("Authorization", f"Bearer {TOKEN}")\n    for attempt in range(6):\n        try:\n            return json.loads(urllib.request.urlopen(req, timeout=30).read())\n        except urllib.error.HTTPError as e:\n            if e.code == 404:\n                return None\n            if e.code in (403, 429):\n                # primary rate limit: wait for the reset the API announces (bounded); else short backoff\n                remaining, reset = e.headers.get("x-ratelimit-remaining"), e.headers.get("x-ratelimit-reset")\n                if remaining == "0" and reset:\n                    wait = max(1, int(reset) - int(time.time()) + 2)\n                    if wait > MAX_RESET_WAIT:\n                        raise RuntimeError(f"GitHub rate limit resets in {wait}s (> {MAX_RESET_WAIT}s); "\n                                           "add a GITHUB_TOKEN Kaggle Secret or rerun later") from e\n                    print(f"rate limited; waiting {wait}s for reset", flush=True)\n                    time.sleep(wait)\n                else:\n                    time.sleep(min(60, 2 ** attempt * 5))\n                continue\n            raise\n    return None\n\n\ndef main():\n    tasks = [json.loads(l) for l in open(sys.argv[1])]\n    meta = {m["instance_id"]: m for m in map(json.loads, open(sys.argv[2]))}\n    with open(sys.argv[3], "w") as ft, open(sys.argv[4], "w") as fm:\n        for t in tasks:\n            m = meta[t["instance_id"]]\n            text, src, created = None, None, None\n            for n in list(m.get("issue_refs", [])) + list(m.get("issue_refs_in_tests", [])):\n                it = gh(f"/repos/{t[\'repo\']}/issues/{n}")\n                if it and "pull_request" not in it:  # a real issue, not a PR\n                    text, src, created = f"{it[\'title\']}\\n{it.get(\'body\') or \'\'}", f"issue#{n}", it["created_at"]\n                    break\n            if text is None and m.get("pr_number"):\n                pr = gh(f"/repos/{t[\'repo\']}/pulls/{m[\'pr_number\']}")\n                if pr:\n                    text, src, created = f"{pr[\'title\']}\\n{pr.get(\'body\') or \'\'}", f"pr_body#{m[\'pr_number\']}", pr["created_at"]\n            if text is not None:\n                t["problem_statement"], t["created_at"] = text.replace("\\r\\n", "\\n"), created\n                m.update(needs_issue_text=False, problem_statement_source=src)\n            ft.write(json.dumps(t) + "\\n")\n            fm.write(json.dumps(m) + "\\n")\n            print(t["instance_id"], src, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote common/github_text.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/common', exist_ok=True)
open('/kaggle/working/pd/common/embed_nodes.py', 'w').write('"""Embed code-graph nodes with an OPEN model and write the harness\'s embeddings .npz.\n\nWhy: the harness adds the "Code Intelligence Tools" prompt section only if both a graph .json and an\nembeddings .npz (>100 bytes) exist for the task (swegemma.harness.agent_runner). The released\nembeddings come from an undisclosed Google-internal model (embedding_utils.DEFAULT_FUNC_EMBEDDINGS_SSTABLE),\nso held-out tasks need a disclosed substitute. Format: npz of {node id: float32[dim]}, allow_pickle=False,\nsame keys as the graph\'s node ids (embedding_utils.load_embeddings_from_npz).\n\nModels (ungated, permissive, NATIVE transformers architectures + safetensors only: no remote code is executed,\nno pickled weights are loaded; models needing trust_remote_code, e.g. CodeT5+ or nomic, are excluded on purpose):\n  gte       Alibaba-NLP/gte-modernbert-base     (Apache-2.0) 768-d, trained incl. code retrieval\n  bge       BAAI/bge-small-en-v1.5              (MIT)        384-d\nOptional --map W.npz: a linear map (d_model+1 x 256) fit by pilot_d/embedding_parity.py that translates the\nopen model\'s space into the released embeddings\' geometry (raw-cosine neighbours, collapsed scale).\n\nUsage: python common/embed_nodes.py GRAPH.json OUT.npz --model gte [--input text|name] [--map W.npz]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nMODELS = {\n    "gte": ("Alibaba-NLP/gte-modernbert-base", None),\n    "bge": ("BAAI/bge-small-en-v1.5", None),\n}\nMAX_CHARS = 2000  # node texts are code; long bodies are truncated (tokenizers truncate anyway)\n\n\ndef node_inputs(graph: dict, mode: str) -> tuple[list[str], list[str]]:\n    ids = [n["id"] for n in graph["nodes"]]\n    if mode == "name":\n        return ids, ids\n    return ids, [f"{n[\'id\']}\\n{(n.get(\'text\') or \'\')[:MAX_CHARS]}" for n in graph["nodes"]]\n\n\ndef embed_texts(model_key: str, texts: list[str], batch: int = 32) -> np.ndarray:\n    import torch\n    from sentence_transformers import SentenceTransformer\n\n    name, mrl = MODELS[model_key]\n    model = SentenceTransformer(name, trust_remote_code=False, truncate_dim=mrl, device="cpu",\n                                model_kwargs={"use_safetensors": True, "dtype": torch.float32})\n    model.max_seq_length = min(model.max_seq_length or 512, 512)\n    return model.encode(texts, batch_size=batch, convert_to_numpy=True,\n                        normalize_embeddings=True).astype(np.float32)\n\n\ndef apply_map(X: np.ndarray, W: np.ndarray) -> np.ndarray:\n    return (np.hstack([X, np.ones((len(X), 1), dtype=X.dtype)]) @ W).astype(np.float32)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("graph")\n    p.add_argument("out")\n    p.add_argument("--model", default="gte", choices=sorted(MODELS))\n    p.add_argument("--input", default="text", choices=["text", "name"])\n    p.add_argument("--map")\n    a = p.parse_args()\n    g = json.load(open(a.graph))\n    ids, texts = node_inputs(g, a.input)\n    X = embed_texts(a.model, texts)\n    if a.map:\n        X = apply_map(X, np.load(a.map)["W"])\n    Path(a.out).parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(a.out, **{i: X[k].astype(np.float32) for k, i in enumerate(ids)})\n    print(a.out, X.shape)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote common/embed_nodes.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/__init__.py', 'w').write(''); print('wrote pilot_d/__init__.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/build_tasks.py', 'w').write('"""Pilot D: turn mined candidates into competition-format tasks (+ sidecar metadata).\n\nWrites, under OUT_DIR (outside git):\n  tasks.jsonl            competition fields (+ optional FAIL_TO_PASS/PASS_TO_PASS later)\n  meta.jsonl             provenance: fix commit, PR, issue refs, dates, AI marker, timings\n  snapshots/<iid>.tgz    dev-layout snapshot\nproblem_statement is left EMPTY with needs_issue_text=true: issue text must be fetched from the\nGitHub API (common/github_text.py), which this cloud session cannot reach for third-party repos.\n\nUsage: python -I pilot_d/build_tasks.py CANDIDATES.jsonl REPO_DIR OUT_DIR [--n N] [--issue-linked-only]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom common.snapshot import build_snapshot  # noqa: E402\n\n\ndef spread(rows, n):\n    """Pick n rows spread evenly over time (rows sorted by commit date)."""\n    rows = sorted(rows, key=lambda r: r["committed_at"])\n    if len(rows) <= n:\n        return rows\n    step = len(rows) / n\n    return [rows[int(i * step)] for i in range(n)]\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("candidates")\n    p.add_argument("repo_dir")\n    p.add_argument("out_dir")\n    p.add_argument("--n", type=int, default=12)\n    p.add_argument("--issue-linked-only", action="store_true")\n    p.add_argument("--ids", nargs="*", help="build exactly these instance_ids (e.g. from pilot_d/manifest.json)")\n    a = p.parse_args()\n    rows = [json.loads(l) for l in open(a.candidates)]\n    if a.ids:\n        chosen = [r for r in rows if r["instance_id"] in set(a.ids)]\n    else:\n        if a.issue_linked_only:\n            rows = [r for r in rows if r["issue_refs"]]\n        chosen = spread(rows, a.n)\n    out = Path(a.out_dir)\n    (out / "snapshots").mkdir(parents=True, exist_ok=True)\n    with open(out / "tasks.jsonl", "a") as ft, open(out / "meta.jsonl", "a") as fm:\n        for r in chosen:\n            t0 = time.perf_counter()\n            snap = build_snapshot(a.repo_dir, r["base_commit"], str(out / "snapshots" / f"{r[\'instance_id\']}.tgz"))\n            task = {\n                "instance_id": r["instance_id"], "repo": r["repo"], "base_commit": r["base_commit"],\n                "patch": r["patch"], "test_patch": r["test_patch"],\n                "problem_statement": "", "hints_text": "", "created_at": r["authored_at"],\n            }\n            meta = {k: r.get(k) for k in ("instance_id", "repo", "fix_commit", "pr_number", "issue_refs",\n                                      "issue_refs_in_tests", "subject",\n                                      "committed_at", "authored_at", "code_files", "test_files",\n                                      "code_lines_changed", "ai_authored_marker")}\n            meta.update(snap, needs_issue_text=True, build_seconds=round(time.perf_counter() - t0, 1))\n            ft.write(json.dumps(task) + "\\n")\n            fm.write(json.dumps(meta) + "\\n")\n            print(r["instance_id"], snap, meta["build_seconds"], flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote pilot_d/build_tasks.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/export_level_a.py', 'w').write('"""Assemble the held-out export in docs/PROBE_INPUT.md "Level A" layout (pilot C\'s probe input).\n\nOUT/\n  tasks.jsonl                 exactly the 8 string fields of PROBE_INPUT (no FAIL_TO_PASS/PASS_TO_PASS: with them the\n                              scorer would use F2P+P2P as required tests instead of the test_patch rule the dev tasks use)\n  snapshots/<id>.tgz          dev layout (common/snapshot.py)\n  graphs/<short>_<base>.json  released-rule graphs (common/graph_build.py faithful mode; generator: codegraph-loc\n                              1e8e0ab + exact released id/text/edge rules; 100% node-id match on 7/7 dev graphs)\n  embeddings/<short>_<base>.npz   disclosed substitute (common/embed_nodes.py; see EXPORT.json "embeddings")\n  wheels/*.whl                test-only deps (cp312 for the Kaggle subprocess host, cp313 for the Docker image)\n  heldout_meta.jsonl          sidecar per task: provenance, F2P/P2P, gold nodes, verified repro test, probe eligibility\n  EXPORT.json                 versions, generator, embedding model, counts, known gaps\nOnly tasks valid in the harness (gold passes, empty patch fails) are exported. Tasks with an empty\nproblem_statement are kept but flagged (needs_issue_text): fill them with common/github_text.py where\napi.github.com is reachable (Kaggle notebook with Internet ON, or a laptop) BEFORE any probe or rollout.\n\nUsage:\n  python pilot_d/export_level_a.py --work WORK/heldout --out OUT [--tasks tasks_text.jsonl] [--meta meta_text.jsonl]\n         [--embed-model gte --embed-input text --embed-map W.npz] [--wheels-req pilot_d/env/*.txt]\n(interpreter with swegemma + codegraph-loc; --embed-* needs sentence-transformers)\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom common.graph_build import build as build_graph  # noqa: E402\n\nFIELDS = ("instance_id", "repo", "base_commit", "patch", "test_patch", "problem_statement", "hints_text", "created_at")\n\n\ndef jl(p: Path) -> list[dict]:\n    return [json.loads(l) for l in open(p)] if p.exists() else []\n\n\ndef required_test_names(test_patch: str) -> list[str]:\n    from swegemma.harness.verification import _extract_test_functions_from_patch\n    return _extract_test_functions_from_patch(test_patch)\n\n\ndef probe_eligibility(task: dict, ann: dict) -> dict:\n    """PROBE_INPUT §3 rule 3, mirrored from pilot_c/select_tasks.eligible (minus its dev-repo filter)."""\n    files = re.findall(r"^\\+\\+\\+ b/(\\S+)", task["patch"], re.M)\n    py = [f for f in files if f.endswith(".py")]\n    new_files = sorted({h["file"] for h in ann.get("edit_hunks", [])\n                        if h.get("old_start") == 0 and h.get("old_len") == 0})\n    changed = sum(1 for l in task["patch"].splitlines() if l[:1] in "+-" and not l.startswith(("+++", "---")))\n    funcs = [s for s in ann.get("gold_nodes", [])] + [s for s in ann.get("gold_nodes_missing_from_graph_rule", [])\n                                                      if "<module>" not in s]\n    names = required_test_names(task["test_patch"])\n    checks = {"one_src_py_file": len(files) == 1 and len(py) == 1, "file_exists_at_base": not new_files,\n              "one_or_two_functions": 1 <= len(funcs) <= 2, "max_40_changed_lines": changed <= 40,\n              "has_required_test_name": len(names) >= 1}\n    return {"eligible": all(checks.values()), "checks": checks, "changed_lines": changed,\n            "n_functions": len(funcs), "required_test_names": names}\n\n\n_EMB_CACHE: dict[str, dict[str, "object"]] = {}\n\n\ndef write_embeddings(gpath: Path, epath: Path, short: str, a) -> None:\n    """Embed one graph\'s nodes, re-using vectors of identical input texts across commits of the same repo\n    (consecutive base commits share almost all nodes, so each new commit costs only its changed nodes)."""\n    import hashlib\n\n    import numpy as np\n\n    from common.embed_nodes import apply_map, embed_texts, node_inputs\n\n    if epath.exists():\n        return\n    cache = _EMB_CACHE.setdefault(short, {})\n    ids, texts = node_inputs(json.load(open(gpath)), a.embed_input)\n    keys = [hashlib.sha1(t.encode("utf-8", "replace")).hexdigest() for t in texts]\n    todo = sorted({k: t for k, t in zip(keys, texts) if k not in cache}.items())\n    if todo:\n        vecs = embed_texts(a.embed_model, [t for _, t in todo])\n        cache.update({k: v for (k, _), v in zip(todo, vecs)})\n    X = np.stack([cache[k] for k in keys])\n    if a.embed_map:\n        X = apply_map(X, np.load(a.embed_map)["W"])\n    np.savez_compressed(epath, **{i: X[n].astype(np.float32) for n, i in enumerate(ids)})\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--work", required=True, help="dir with tasks.jsonl, meta.jsonl, snapshots/, val_subprocess.jsonl, "\n                                                 "testmap.jsonl, annotations.jsonl")\n    p.add_argument("--out", required=True)\n    p.add_argument("--tasks", default="tasks.jsonl")\n    p.add_argument("--meta", default="meta.jsonl")\n    p.add_argument("--validation", default="val_subprocess.jsonl")\n    p.add_argument("--embed-model")\n    p.add_argument("--embed-input", default="text")\n    p.add_argument("--embed-map")\n    p.add_argument("--wheels-req", nargs="*", default=[])\n    p.add_argument("--symlink-snapshots", action="store_true", help="local use: link instead of copying")\n    a = p.parse_args()\n    W, O = Path(a.work), Path(a.out)\n    for d in ("snapshots", "graphs", "embeddings", "wheels"):\n        (O / d).mkdir(parents=True, exist_ok=True)\n    valid = {r["instance_id"] for r in jl(W / a.validation) if r.get("valid")}\n    meta = {m["instance_id"]: m for m in jl(W / a.meta)}\n    tm = {r["instance_id"]: r for r in jl(W / "testmap.jsonl")}\n    ann = {r["instance_id"]: r for r in jl(W / "annotations.jsonl")}\n    tasks = [t for t in jl(W / a.tasks) if t["instance_id"] in valid]\n    commits = {}\n    with open(O / "tasks.jsonl", "w") as ft, open(O / "heldout_meta.jsonl", "w") as fm:\n        for t in tasks:\n            iid = t["instance_id"]\n            ft.write(json.dumps({k: str(t.get(k) or "") for k in FIELDS}) + "\\n")\n            src, dst = W / "snapshots" / f"{iid}.tgz", O / "snapshots" / f"{iid}.tgz"\n            if not dst.exists():\n                (dst.symlink_to(src.resolve()) if a.symlink_snapshots else shutil.copy2(src, dst))\n            commits.setdefault((t["repo"].split("/")[1].lower(), t["base_commit"]), iid)\n            m, A = meta.get(iid, {}), ann.get(iid, {})\n            fm.write(json.dumps({\n                "instance_id": iid, "fix_commit": m.get("fix_commit"), "pr_number": m.get("pr_number"),\n                "issue_refs": sorted(set((m.get("issue_refs") or []) + (m.get("issue_refs_in_tests") or []))),\n                "problem_statement_source": m.get("problem_statement_source", "pending"),\n                "needs_issue_text": not (t.get("problem_statement") or "").strip(),\n                "committed_at": m.get("committed_at"), "ai_authored_marker": m.get("ai_authored_marker"),\n                "FAIL_TO_PASS": tm.get(iid, {}).get("FAIL_TO_PASS", []),\n                "PASS_TO_PASS": tm.get(iid, {}).get("PASS_TO_PASS", []),\n                "gold_nodes": A.get("gold_nodes", []), "edit_hunks": A.get("edit_hunks", []),\n                "added_symbols": A.get("added_symbols", []),\n                "repro_test": A.get("repro_test"), "probe": probe_eligibility(t, A),\n            }) + "\\n")\n    # graphs (+ embeddings) once per (repo, base_commit)\n    for (short, base), iid in commits.items():\n        gpath = O / "graphs" / f"{short}_{base}.json"\n        if not gpath.exists():\n            with tempfile.TemporaryDirectory() as td:\n                with tarfile.open(W / "snapshots" / f"{iid}.tgz") as tf:\n                    tf.extractall(td, filter="data")\n                build_graph(td, short, base, str(O / "graphs"))\n        if a.embed_model:\n            write_embeddings(gpath, O / "embeddings" / f"{short}_{base}.npz", short, a)\n    for req in a.wheels_req:  # test-only deps, both interpreters used by the two sandbox backends\n        if not any(l.split("#")[0].strip() for l in open(req)):\n            continue  # this repo declares no test-only deps\n        for pyv in ("3.12", "3.13"):\n            pip = [shutil.which("pip")] if shutil.which("pip") else [sys.executable, "-m", "pip"]\n            subprocess.run([*pip, "download", "-q", "--no-deps", "-r", req, "--only-binary=:all:",\n                            "--python-version", pyv, "--platform", "manylinux2014_x86_64", "--platform",\n                            "manylinux_2_17_x86_64", "--platform", "manylinux_2_28_x86_64", "-d", str(O / "wheels")],\n                           check=True)\n    rows = jl(O / "heldout_meta.jsonl")\n    info = {\n        "format": "docs/PROBE_INPUT.md v2 Level A (paper/main)", "tasks": len(rows),\n        "repos": sorted({t["repo"] for t in tasks}),\n        "probe_eligible": sum(r["probe"]["eligible"] for r in rows),\n        "needs_issue_text": sum(r["needs_issue_text"] for r in rows),\n        "graph_generator": "pilot/d-heldout common/graph_build.py (faithful) on codegraph-loc 1e8e0ab",\n        "embeddings": ({"model": a.embed_model, "input": a.embed_input, "map": bool(a.embed_map),\n                        "note": "disclosed open-model substitute; released dev embeddings come from an undisclosed "\n                                "Google-internal model. Present so the harness shows the same \'Code Intelligence "\n                                "Tools\' prompt section as on dev tasks."} if a.embed_model else None),\n        "test_dep_wheels": sorted(x.name for x in (O / "wheels").glob("*.whl")),\n        "subprocess_backend_note": "the subprocess sandbox inherits the HOST site-packages: pip install the wheels/ "\n                                   "matching the host Python into the notebook before running the harness",\n    }\n    (O / "EXPORT.json").write_text(json.dumps(info, indent=1))\n    print(json.dumps({k: info[k] for k in ("tasks", "repos", "probe_eligible", "needs_issue_text")}))\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote pilot_d/export_level_a.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/embedding_parity.py', 'w').write('"""Choose and validate the substitute embeddings for held-out tasks (missing-embeddings confound).\n\nOn dev graphs where the released (undisclosed-model) embeddings exist, measure how closely each open\nsubstitute reproduces what the agent would see from `search_similar_code` (harness = raw cosine on the\nstored vectors, top-k):\n  overlap@10   mean |top10_released ∩ top10_substitute| / 10 over all query nodes (random ≈ 10/N)\n  spearman     rank correlation of raw-cosine similarities on 20k random node pairs\n  top1_sim     median raw-cosine similarity of the top-1 neighbour (the number the agent is shown)\nVariants: raw open-model vectors, and the same vectors passed through a linear map into the released\ngeometry, fit by ridge regression with LEAVE-ONE-REPO-OUT evaluation (the scored repo never fits its map).\n\nUsage: python pilot_d/embedding_parity.py OUT.json CACHE_DIR GRAPH1.json:EMB1.npz [GRAPH2.json:EMB2.npz ...]\nWrites aggregates only (no node ids). The final map (fit on all given repos) is saved to CACHE_DIR/W_<model>_<input>.npz;\nit is derived from competition embeddings, so it is not committed: rebuild it with this script.\n"""\nfrom __future__ import annotations\n\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom common.embed_nodes import apply_map, embed_texts, node_inputs  # noqa: E402\n\nCANDIDATES = [("gte", "text"), ("gte", "name"), ("bge", "text")]\nif __import__("os").environ.get("PARITY_CANDIDATES"):  # e.g. "gte/text,bge/text" (the D0 notebook fits only the chosen one)\n    CANDIDATES = [tuple(c.split("/")) for c in __import__("os").environ["PARITY_CANDIDATES"].split(",")]\nLAMBDAS = (0.1, 1.0, 10.0, 100.0)\n\n\ndef raw_cos(M):\n    n = np.linalg.norm(M, axis=1, keepdims=True)\n    n[n == 0] = 1\n    U = M / n\n    return U @ U.T\n\n\ndef metrics(R, C, rng):\n    SR, SC = raw_cos(R.astype(np.float64)), raw_cos(C.astype(np.float64))\n    np.fill_diagonal(SR, -np.inf); np.fill_diagonal(SC, -np.inf)\n    k = 10\n    tr, tc = np.argsort(-SR, axis=1)[:, :k], np.argsort(-SC, axis=1)[:, :k]\n    ov = np.mean([len(set(a) & set(b)) / k for a, b in zip(tr, tc)])\n    n = len(R)\n    i, j = rng.integers(0, n, 20000), rng.integers(0, n, 20000)\n    keep = i != j\n    from scipy.stats import spearmanr  # noqa: PLC0415\n    sp = spearmanr(SR[i[keep], j[keep]], SC[i[keep], j[keep]]).statistic\n    return {"overlap@10": round(float(ov), 4), "random@10": round(10 / (n - 1), 4), "spearman": round(float(sp), 4),\n            "top1_sim_substitute": round(float(np.median(SC.max(1))), 4),\n            "top1_sim_released": round(float(np.median(SR.max(1))), 4)}\n\n\ndef fit_map(Xs, Ys, lam):\n    X = np.vstack([np.hstack([x, np.ones((len(x), 1))]) for x in Xs]).astype(np.float64)\n    Y = np.vstack(Ys).astype(np.float64)\n    A = X.T @ X + lam * len(X) / 1000 * np.eye(X.shape[1])\n    A[-1, -1] -= lam * len(X) / 1000  # do not shrink the bias row\n    return np.linalg.solve(A, X.T @ Y)\n\n\ndef main():\n    out, cache = sys.argv[1], Path(sys.argv[2])\n    cache.mkdir(parents=True, exist_ok=True)\n    pairs = [a.split(":") for a in sys.argv[3:]]\n    rng = np.random.default_rng(0)\n    data = []\n    for gp, ep in pairs:\n        g = json.load(open(gp))\n        z = np.load(ep, allow_pickle=False)\n        ids = [n["id"] for n in g["nodes"] if n["id"] in z.files]\n        data.append({"repo": Path(gp).stem.split("_")[0], "graph": g, "ids": ids,\n                     "R": np.stack([z[i] for i in ids]).astype(np.float32)})\n    report = {"what": __doc__.split("\\n")[0], "repos": [d["repo"] for d in data],\n              "nodes": [len(d["ids"]) for d in data], "candidates": {}}\n    for model, mode in CANDIDATES:\n        X = []\n        for d in data:\n            f = cache / f"{d[\'repo\']}_{len(d[\'ids\'])}_{model}_{mode}.npy"\n            if not f.exists():\n                idx = {i: k for k, i in enumerate(n["id"] for n in d["graph"]["nodes"])}\n                all_ids, texts = node_inputs(d["graph"], mode)\n                E = embed_texts(model, texts)\n                np.save(f, E[[idx[i] for i in d["ids"]]])\n            X.append(np.load(f))\n        res = {"raw": [], "mapped_lorocv": []}\n        for t, d in enumerate(data):\n            res["raw"].append(metrics(d["R"], X[t], rng) if X[t].shape[1] else None)\n            # leave-one-repo-out map; lambda chosen by inner leave-one-out over the training repos\n            tr = [s for s in range(len(data)) if s != t]\n            best = None\n            for lam in LAMBDAS:\n                inner = []\n                for v in tr:\n                    fit = [s for s in tr if s != v]\n                    W = fit_map([X[s] for s in fit], [data[s]["R"] for s in fit], lam)\n                    inner.append(metrics(data[v]["R"], apply_map(X[v], W), rng)["overlap@10"])\n                score = float(np.mean(inner))\n                best = max(best or (score, lam), (score, lam))\n            W = fit_map([X[s] for s in tr], [data[s]["R"] for s in tr], best[1])\n            m = metrics(d["R"], apply_map(X[t], W), rng)\n            m["lambda"] = best[1]\n            res["mapped_lorocv"].append(m)\n        for key in ("raw", "mapped_lorocv"):\n            res[key + "_mean"] = {k: round(float(np.mean([r[k] for r in res[key]])), 4)\n                                  for k in ("overlap@10", "random@10", "spearman", "top1_sim_substitute")}\n        res["top1_sim_released_mean"] = round(float(np.mean([r["top1_sim_released"] for r in res["raw"]])), 4)\n        lam_all = max(set(r["lambda"] for r in res["mapped_lorocv"]), key=[r["lambda"] for r in res["mapped_lorocv"]].count)\n        np.savez(cache / f"W_{model}_{mode}.npz", W=fit_map(X, [d["R"] for d in data], lam_all).astype(np.float32))\n        report["candidates"][f"{model}/{mode}"] = res\n        print(model, mode, "raw", res["raw_mean"], "| mapped", res["mapped_lorocv_mean"], flush=True)\n    Path(out).write_text(json.dumps(report, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote pilot_d/embedding_parity.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/check_prompt_parity.py', 'w').write('"""Check the missing-embeddings confound is fixed: same prompt section and working graph tools as dev tasks.\n\nFor every task of an export (Level A layout) this calls the harness\'s OWN code:\n  * swegemma.harness.agent_runner.build_agent_prompt with the sample agent\'s declared graph tools, once with the\n    export\'s embeddings dir and once with an empty one: is "## Code Intelligence Tools" present?\n  * SwegemmaContext(...).create_tools() offline (no sandbox): search_similar_code and get_code_neighbors on the\n    task\'s first gold node (from heldout_meta.jsonl): status ok and non-empty results?\nPrints aggregate counts; --dev TASKS.jsonl GRAPHS EMB runs the same check on dev tasks for comparison.\n\nUsage: python pilot_d/check_prompt_parity.py EXPORT_DIR [--out RESULT.json]\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport tempfile\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nSECTION = "## Code Intelligence Tools"\nGRAPH_TOOLS = ("search_similar_code", "get_code_neighbors", "get_code_subgraph")\n\n\ndef tool_calls(task, node: str, graph_dir: str, emb_dir: str) -> dict:\n    from swegemma.context import SwegemmaContext\n\n    ctx = SwegemmaContext(docker_manager=None, task=task, repo=task.repo, graph_dir=graph_dir, embeddings_dir=emb_dir)\n    tools = ctx.create_tools()\n    out = {}\n    for name, args in (("search_similar_code", {"query": node}), ("get_code_neighbors", {"node": node})):\n        fn = next((f for k, f in tools.items() if name in k), None)\n        try:\n            r = fn(**args)\n            r = json.loads(r) if isinstance(r, str) else r\n            items = r.get("results") or r.get("neighbors") or r.get("similar_nodes") or r.get("matches") or []\n            out[name] = {"ok": r.get("status") == "ok" and len(items) > 0, "n": len(items),\n                         "keys": sorted(r)[:6]}\n        except Exception as e:  # report, do not hide\n            out[name] = {"ok": False, "error": f"{type(e).__name__}: {e}"[:120]}\n    return out\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("export")\n    p.add_argument("--out")\n    a = p.parse_args()\n    from swegemma.harness.agent_runner import build_agent_prompt\n    from swegemma.models import load_tasks\n\n    E = Path(a.export)\n    tasks = load_tasks(E / "tasks.jsonl")\n    meta = {json.loads(l)["instance_id"]: json.loads(l) for l in open(E / "heldout_meta.jsonl")}\n    empty = tempfile.mkdtemp()\n    rows = []\n    for t in tasks:\n        cfg = SimpleNamespace(graph_dir=str(E / "graphs"), embeddings_dir=str(E / "embeddings"),\n                              enable_sandbox_testing=True, budget=None)\n        with_emb = SECTION in build_agent_prompt(t, cfg, "", declared_tools=GRAPH_TOOLS)\n        cfg.embeddings_dir = empty\n        without = SECTION in build_agent_prompt(t, cfg, "", declared_tools=GRAPH_TOOLS)\n        gold = (meta.get(t.instance_id, {}).get("gold_nodes") or [None])[0]\n        tc = tool_calls(t, gold, str(E / "graphs"), str(E / "embeddings")) if gold and with_emb else {}\n        rows.append({"instance_id": t.instance_id, "section_with_embeddings": with_emb,\n                     "section_without_embeddings": without, "tools": tc})\n    agg = {"tasks": len(rows),\n           "section_present_with_embeddings": sum(r["section_with_embeddings"] for r in rows),\n           "section_present_without_embeddings": sum(r["section_without_embeddings"] for r in rows),\n           "search_similar_code_ok": sum(1 for r in rows if r["tools"].get("search_similar_code", {}).get("ok")),\n           "get_code_neighbors_ok": sum(1 for r in rows if r["tools"].get("get_code_neighbors", {}).get("ok")),\n           "tool_errors": sorted({v.get("error") for r in rows for v in r["tools"].values() if v.get("error")})[:5]}\n    print(json.dumps(agg, indent=1))\n    if a.out:\n        Path(a.out).write_text(json.dumps({"aggregate": agg, "rows": rows}, indent=1))\n\n\nif __name__ == "__main__":\n    main()\n'); print('wrote pilot_d/check_prompt_parity.py')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d', exist_ok=True)
open('/kaggle/working/pd/pilot_d/manifest.json', 'w').write('{\n "since": "2026-05-01",\n "clone_shallow_since": "2026-03-15",\n "note": "Held-out pool (pilot D). instance_id = <repo_short>_<PR number or fix-commit prefix>; public upstream commits only. Rebuilt by notebooks/D0_heldout_export__CPU.ipynb (pilot_d/build_tasks.py --ids).",\n "selection_rule": "sqlglot: fix commits with an issue reference in the message or test diff; other repos: all fix commits passing common/mining.py filters (<=3 files, <=200 changed lines)",\n "dropped_repos": {\n  "pydata/xarray": "1/12 valid locally: whole test modules skipped for missing optional backends, version-sensitive failures; environment surface too large",\n  "Python-Markdown/markdown": "extensions load by entry-point name, which needs an INSTALLED copy; the subprocess sandbox never installs the workspace (2/9 valid)"\n },\n "embeddings": {\n  "model": "bge",\n  "input": "text",\n  "map": false,\n  "why": "top-10-neighbour agreement with the released embeddings on 4 dev graphs: bge 0.174, gte 0.183 (random 0.009); bge chosen for speed (22 vs 5 texts/s on 4 CPUs, measured), needed to embed ~80 graphs on Kaggle CPU; the linear map into the released geometry lowered agreement for every model (leave-one-repo-out), so none is used"\n },\n "repos": [\n  {\n   "short": "sqlglot",\n   "repo": "tobymao/sqlglot",\n   "url": "https://github.com/tobymao/sqlglot",\n   "pkg": "sqlglot",\n   "licence": "MIT",\n   "instance_ids": [\n    "sqlglot_8856",\n    "sqlglot_8857",\n    "sqlglot_8825",\n    "sqlglot_8809",\n    "sqlglot_8686",\n    "sqlglot_8670",\n    "sqlglot_8666",\n    "sqlglot_8596",\n    "sqlglot_a00dc2e5c4",\n    "sqlglot_8523",\n    "sqlglot_f8606fe8f9",\n    "sqlglot_8538",\n    "sqlglot_b65592b458",\n    "sqlglot_8476",\n    "sqlglot_8447",\n    "sqlglot_8434",\n    "sqlglot_981e492413",\n    "sqlglot_8d9f5d08cc",\n    "sqlglot_a3884168f6",\n    "sqlglot_8369",\n    "sqlglot_3ca824895e",\n    "sqlglot_8321",\n    "sqlglot_8284",\n    "sqlglot_8285",\n    "sqlglot_8269",\n    "sqlglot_8267",\n    "sqlglot_8270",\n    "sqlglot_8260",\n    "sqlglot_40f285cb61",\n    "sqlglot_a570cf8a90",\n    "sqlglot_f64adb4c9a",\n    "sqlglot_83129c711b",\n    "sqlglot_8138",\n    "sqlglot_2f914c54aa",\n    "sqlglot_c25225d9ac",\n    "sqlglot_df5ec1ecd4",\n    "sqlglot_b4776a4099",\n    "sqlglot_d6d174e85e",\n    "sqlglot_7968",\n    "sqlglot_169b8364b6",\n    "sqlglot_a1e3338e31",\n    "sqlglot_a66104f52a",\n    "sqlglot_7809",\n    "sqlglot_4c2cd15c7a",\n    "sqlglot_6ae4b49f1d",\n    "sqlglot_a40dde9d39",\n    "sqlglot_7640",\n    "sqlglot_a1fa1546c9",\n    "sqlglot_b9f18eea67",\n    "sqlglot_17432c6117"\n   ],\n   "fix_commits": {\n    "sqlglot_8856": "6de2df31b0ae8b21155828f575a14dc8903c1b9b",\n    "sqlglot_8857": "f70f4e5be5acc50058f4d4ac1ee0f250e203916b",\n    "sqlglot_8825": "13a570c6f790a6140c9c9ae004914b081beb7ae6",\n    "sqlglot_8809": "2d4b5a5b0aea46e7f5a4fbe09c1d60fcafad76b3",\n    "sqlglot_8686": "fdd0e9a464f0f9ec425c56ae83fc87a5e2052087",\n    "sqlglot_8670": "f8c314c449080860604b9ddd716d6bc75c89155f",\n    "sqlglot_8666": "513beedd3753a22c1ce234059bd5d831f64e48f0",\n    "sqlglot_8596": "e38929edd78a20f74fc19fee819e14ac23add687",\n    "sqlglot_a00dc2e5c4": "a00dc2e5c489214bea53bacbbeba8713f21ebce1",\n    "sqlglot_8523": "3096a591d65b1b00ba9abdf482fc7b7973a6e6af",\n    "sqlglot_f8606fe8f9": "f8606fe8f97a10a3ba00f42104836add1299966f",\n    "sqlglot_8538": "e1e0d2e52f45247bfa0ab815eb8a7388ea50a0e8",\n    "sqlglot_b65592b458": "b65592b45831df5e7f13d418e50f24c4cea118f0",\n    "sqlglot_8476": "ac084c5bdb2f188ff878687000273ae2247db24b",\n    "sqlglot_8447": "b3b6c57cea516bf8f76c6262b308d86dff82ca1b",\n    "sqlglot_8434": "d2fbb215ff6bd548547a151e8838c277aad076c1",\n    "sqlglot_981e492413": "981e492413740f7740d2b0288e3da5b4028c767e",\n    "sqlglot_8d9f5d08cc": "8d9f5d08cce4b72b4df1a8f6394c2aa043cc7de8",\n    "sqlglot_a3884168f6": "a3884168f639633fa8f64334bf63e4b343bedfe2",\n    "sqlglot_8369": "65eb23b53420212c20f394d399ca18efefec90e3",\n    "sqlglot_3ca824895e": "3ca824895ef423f7895fb13d72357548c0f1f367",\n    "sqlglot_8321": "e27b7654a9d710e29da6f42994de845a99444c84",\n    "sqlglot_8284": "6b9cb87bd01ed4606d1902a0266ff79aad78b54f",\n    "sqlglot_8285": "67cf1ec91dfe2658f41c049130ab242a40bf4789",\n    "sqlglot_8269": "d0aa2b432498ec00f88a8721a904afe2e1b5c24d",\n    "sqlglot_8267": "3110e151bfe6e229ec6b85708a56a9340a56c6a6",\n    "sqlglot_8270": "f131ec9f9ea40ce55f0e254baa41b01f87bad25c",\n    "sqlglot_8260": "88419b1aa19453be7fca80c0178362b1fbf2d6e1",\n    "sqlglot_40f285cb61": "40f285cb61edff4a40950fc708e79ae4e8ed583d",\n    "sqlglot_a570cf8a90": "a570cf8a908d02e78b66cec1478ab066f1ff1f0d",\n    "sqlglot_f64adb4c9a": "f64adb4c9a3f96ff4ad692d571b80848ad1b649d",\n    "sqlglot_83129c711b": "83129c711baf600f6ae902f79cf1cd6e50bdba58",\n    "sqlglot_8138": "1dae76b5833a47c9e2bc0a72a93d94d829377a0c",\n    "sqlglot_2f914c54aa": "2f914c54aa4817a0a5e65a0d7d5e7cba9037ac30",\n    "sqlglot_c25225d9ac": "c25225d9ac7390ef87bd02a3d2ac005073a1e27d",\n    "sqlglot_df5ec1ecd4": "df5ec1ecd43b83ac63800bf40b04362c9af0173e",\n    "sqlglot_b4776a4099": "b4776a40995985c5523849269e1ab6d29e667ec6",\n    "sqlglot_d6d174e85e": "d6d174e85e37d7644d16c0bce3e4fb49384a6bc8",\n    "sqlglot_7968": "06eb0510b06307ffad115827bc7c9c53deecb216",\n    "sqlglot_169b8364b6": "169b8364b643c4285dcf073aa2f357c5b40831a6",\n    "sqlglot_a1e3338e31": "a1e3338e31354496a3ef3bed5f3c5e0c4df4e0f6",\n    "sqlglot_a66104f52a": "a66104f52aca450087524f84b76e29173971e656",\n    "sqlglot_7809": "cf7e4bf4a57d331e4a6e5f4933cb7daf0e503c3b",\n    "sqlglot_4c2cd15c7a": "4c2cd15c7a66bb240a8508517bb492aa7ed6c84a",\n    "sqlglot_6ae4b49f1d": "6ae4b49f1d2ef0c4f30575a29a0d1b3273520abf",\n    "sqlglot_a40dde9d39": "a40dde9d3992b2b1df4f4ff5dccde240d618dabd",\n    "sqlglot_7640": "63296846f6a7a683af21683acb24aad7a63bb3f5",\n    "sqlglot_a1fa1546c9": "a1fa1546c926b2858cfc5adcfbd186731de40eba",\n    "sqlglot_b9f18eea67": "b9f18eea6762f8ce03ff5a426db8195db375f455",\n    "sqlglot_17432c6117": "17432c61175194ee0f56f0703393e8c5fdc9e81f"\n   }\n  },\n  {\n   "short": "networkx",\n   "repo": "networkx/networkx",\n   "url": "https://github.com/networkx/networkx",\n   "pkg": "networkx",\n   "licence": "BSD-3-Clause",\n   "instance_ids": [\n    "networkx_8931",\n    "networkx_8906",\n    "networkx_8892",\n    "networkx_8890",\n    "networkx_8895",\n    "networkx_8899",\n    "networkx_8883",\n    "networkx_8882",\n    "networkx_8881",\n    "networkx_8880",\n    "networkx_8879",\n    "networkx_8620",\n    "networkx_8837",\n    "networkx_8813",\n    "networkx_8800",\n    "networkx_8808",\n    "networkx_8807",\n    "networkx_8750",\n    "networkx_8734",\n    "networkx_8726",\n    "networkx_8659"\n   ],\n   "fix_commits": {\n    "networkx_8931": "c1ebe046ec672eca7cba4dc88d6e0145406d693a",\n    "networkx_8906": "9f6079c0120416599ea5d4018bbaba229ea2e456",\n    "networkx_8892": "083e47255d7087340e97ec13d256e9241ec25e77",\n    "networkx_8890": "c94928ed94899033126c9d47f797a1f698584b20",\n    "networkx_8895": "56c7b0d0b6141ec0ffa6a9ebb5494417120b7f50",\n    "networkx_8899": "31ab6fd7e1e149e887282b83a91752c486d071b6",\n    "networkx_8883": "0db8227000872d7a9f6ce84c54ba1e5e99429122",\n    "networkx_8882": "8e0b38e1a11335f07db13a13a6a5955eecf33408",\n    "networkx_8881": "c279073cc33dab38777e701370210d0a7a98849a",\n    "networkx_8880": "0e3f58edd891bbef8f8013f147e8b679deba9970",\n    "networkx_8879": "73cb92bbaed12828848f2609340442119e7e0fe8",\n    "networkx_8620": "0c0c31bc54c563bb31771ef6420454a697aed12c",\n    "networkx_8837": "416c3e26da0567c9242aa415810dcffca27d625f",\n    "networkx_8813": "2aecdbdd18210c38dc2b9fe32da3825bf30a3c33",\n    "networkx_8800": "139259c6c5479b35ddde9318203ae715b25a3192",\n    "networkx_8808": "a5652a9d440b6d98ec7c88761834adc293680d42",\n    "networkx_8807": "2ebe5f80a114a181e03e18acc9491e45346598c0",\n    "networkx_8750": "b1f956344071ba1adfad17d56d62a7a78dfd37e6",\n    "networkx_8734": "d8081795e1344c576590b214c2e4f264b5e87244",\n    "networkx_8726": "e9eac8a7e04df0b9cdcff8ed1682b3971678f4ed",\n    "networkx_8659": "e187ba20b38583a329c960062e4b9e0f23c8fc74"\n   }\n  },\n  {\n   "short": "sympy",\n   "repo": "sympy/sympy",\n   "url": "https://github.com/sympy/sympy",\n   "pkg": "sympy",\n   "licence": "BSD-3-Clause",\n   "instance_ids": [\n    "sympy_30866",\n    "sympy_30141",\n    "sympy_30841",\n    "sympy_30646",\n    "sympy_30768",\n    "sympy_30716",\n    "sympy_29504",\n    "sympy_30591",\n    "sympy_30590",\n    "sympy_30567",\n    "sympy_30515",\n    "sympy_30476",\n    "sympy_30562",\n    "sympy_30514"\n   ],\n   "fix_commits": {\n    "sympy_30866": "a11605ca930cc3138cdd6c90bc8179e9e166dd8b",\n    "sympy_30141": "17594ab1954648db1238cb1372c50b479b70fc73",\n    "sympy_30841": "f8ea58b03ec08777ed7832bf85606c1fae9f935a",\n    "sympy_30646": "65c7920cb5e5c8958872c11f3ce1b3ba6c3e3395",\n    "sympy_30768": "12a149c2196ea036e3115fa723b20ed311499db3",\n    "sympy_30716": "bcd3a171919e08f940d5e97b165b5191f37ed901",\n    "sympy_29504": "2c89c30012477411849aa590861dfcae7d1e11f4",\n    "sympy_30591": "a1021ffb1f40f860960119e70e6329cd5937862c",\n    "sympy_30590": "94f14043f5195c6c58d2c21dcc2beccd9d476aa4",\n    "sympy_30567": "5c5ec9b873a29b2fdc8dc6759bb9a56f6b3d69c4",\n    "sympy_30515": "f14b5cd2d13c0145521eccec991c315794952117",\n    "sympy_30476": "ee699d48c852e1e98e719870c0dab6905e8326ed",\n    "sympy_30562": "5968b9bfd2bd3db4b6caf35bc07712cb70b8807b",\n    "sympy_30514": "31e03dc7ed8114117274ea679be1835044cec9c9"\n   }\n  },\n  {\n   "short": "pyparsing",\n   "repo": "pyparsing/pyparsing",\n   "url": "https://github.com/pyparsing/pyparsing",\n   "pkg": "pyparsing",\n   "licence": "MIT",\n   "instance_ids": [\n    "pyparsing_653",\n    "pyparsing_652",\n    "pyparsing_a6ae98ba59",\n    "pyparsing_92f52e2c3b",\n    "pyparsing_9b728b8835",\n    "pyparsing_e378aaf45c",\n    "pyparsing_16f348651c",\n    "pyparsing_73a8b553fb",\n    "pyparsing_621"\n   ],\n   "fix_commits": {\n    "pyparsing_653": "efd56db4e59f36b1673ce0eb0823e3afaa9d1201",\n    "pyparsing_652": "51560caba558d87349f4f497b82acef1de9ec8f8",\n    "pyparsing_a6ae98ba59": "a6ae98ba59b993d0b6d0475e7cb3b0a33c853f22",\n    "pyparsing_92f52e2c3b": "92f52e2c3b3ffb24215e966c0c349ca5678a7f86",\n    "pyparsing_9b728b8835": "9b728b8835aeef5974f0ee2be8df204b642ba008",\n    "pyparsing_e378aaf45c": "e378aaf45c5fad96d69daac25f53ff7469cb2878",\n    "pyparsing_16f348651c": "16f348651c167f7d557230ecbdf5a678c62e72aa",\n    "pyparsing_73a8b553fb": "73a8b553fb5fc50b3cb2e4b5b2b8652f8d997a29",\n    "pyparsing_621": "0a2d906a3e6b9a30b6a0701588caec3937ff271f"\n   }\n  }\n ]\n}'); print('wrote pilot_d/manifest.json')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d/env', exist_ok=True)
open('/kaggle/working/pd/pilot_d/env/_harness.txt', 'w').write('# What the harness itself needs in the host env to run verification (Dockerfile.sandbox pins pytest-timeout 2.1.0)\npytest==9.1.1\npytest-timeout==2.1.0\n'); print('wrote pilot_d/env/_harness.txt')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d/env', exist_ok=True)
open('/kaggle/working/pd/pilot_d/env/networkx.txt', 'w').write('# Test-only dependencies of networkx/networkx used via pytest.importorskip (requirements/test.txt + extras)\nnumpy==2.3.4\nscipy==1.16.3\npandas==2.3.3\n'); print('wrote pilot_d/env/networkx.txt')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d/env', exist_ok=True)
open('/kaggle/working/pd/pilot_d/env/pyparsing.txt', 'w').write('# pyparsing/pyparsing: no test-only dependencies\n'); print('wrote pilot_d/env/pyparsing.txt')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d/env', exist_ok=True)
open('/kaggle/working/pd/pilot_d/env/sqlglot.txt', 'w').write('# Test-only dependencies of tobymao/sqlglot (from setup.py extras "dev", imported by tests/)\nduckdb==1.4.1\npandas==2.3.3\npytz==2025.2\npython-dateutil==2.9.0.post0\nsix==1.17.0\ntzdata==2025.2\n'); print('wrote pilot_d/env/sqlglot.txt')

In [ ]:
import os; os.makedirs('/kaggle/working/pd/pilot_d/env', exist_ok=True)
open('/kaggle/working/pd/pilot_d/env/sympy.txt', 'w').write('# Test-only dependencies of sympy/sympy (install_requires + sympy/conftest.py requires hypothesis)\nmpmath==1.3.0\nhypothesis==6.140.2\nattrs==25.4.0\nsortedcontainers==2.4.0\n'); print('wrote pilot_d/env/sympy.txt')

## 1. Rebuild the pool from the manifest (public GitHub)

In [ ]:
os.chdir(PD)
man = json.load(open('pilot_d/manifest.json'))
DRY = os.environ.get('D0_DRYRUN') == '1'   # local papermill dry run only: 2 tasks per repo, no GitHub API
if DRY:
    keep = os.environ.get('D0_DRYRUN_REPOS')  # e.g. "pyparsing,networkx" to keep the dry run short
    man['repos'] = [r for r in man['repos'] if not keep or r['short'] in keep.split(',')]
    for r in man['repos']: r['instance_ids'] = r['instance_ids'][:2]
    print('DRY RUN: 2 tasks per repo', [r['short'] for r in man['repos']])
t0 = time.time()
for repo in man['repos']:
    d = W / 'src' / repo['short']
    if not d.exists():
        subprocess.run(['git', 'clone', '-q', f"--shallow-since={man['clone_shallow_since']}", repo['url'], str(d)], check=True)
    cand = H / f"cand_{repo['short']}.jsonl"
    subprocess.run([sys.executable, 'common/mining.py', str(d), '--repo', repo['repo'], '--pkg', repo['pkg'],
                    '--since', man['since'], '--out', str(cand), '--funnel', str(H / f"funnel_{repo['short']}.json")], check=True)
    subprocess.run([sys.executable, 'pilot_d/build_tasks.py', str(cand), str(d), str(H), '--ids', *repo['instance_ids']], check=True)
n = sum(1 for _ in open(H / 'tasks.jsonl')); print('built', n, 'of', sum(len(r['instance_ids']) for r in man['repos']), 'minutes', round((time.time() - t0) / 60, 1))

## 2. Declared test-only deps (installed only if missing; versions recorded)

In [ ]:
import importlib, importlib.metadata as md
used = {}
for env in sorted(glob.glob('pilot_d/env/*.txt')):
    for line in open(env):
        line = line.split('#')[0].strip()
        if not line: continue
        dist = line.split('==')[0]; mod = {'python-dateutil': 'dateutil', 'pytest-timeout': 'pytest_timeout'}.get(dist, dist)
        try: importlib.import_module(mod)
        except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', line], check=True)
        used[dist] = md.version(dist)
print(used)

## 3. Validate in this image (subprocess backend = the scorer's/starter's)

In [ ]:
t0 = time.time()
subprocess.run([sys.executable, 'common/validate.py', '--tasks', str(H / 'tasks.jsonl'), '--snapshots', str(H / 'snapshots'),
                '--backend', 'subprocess', '--concurrency', '4', '--out', str(H / 'val_subprocess.jsonl')], check=True)
val = [json.loads(l) for l in open(H / 'val_subprocess.jsonl')]
valid = sorted(v['instance_id'] for v in val if v['valid'])
print('valid', len(valid), 'of', len(val), 'minutes', round((time.time() - t0) / 60, 1))

## 4. Issue text for valid tasks (GitHub API)

In [ ]:
vs = set(valid)
with open(H / 'tasks_valid.jsonl', 'w') as ft, open(H / 'meta_valid.jsonl', 'w') as fm:
    for l in open(H / 'tasks.jsonl'):
        if json.loads(l)['instance_id'] in vs: ft.write(l)
    for l in open(H / 'meta.jsonl'):
        if json.loads(l)['instance_id'] in vs: fm.write(l)
if DRY:  # this session cannot reach api.github.com; statements stay empty and are flagged needs_issue_text
    shutil.copy(H / 'tasks_valid.jsonl', H / 'tasks_text.jsonl'); shutil.copy(H / 'meta_valid.jsonl', H / 'meta_text.jsonl')
else:
    subprocess.run([sys.executable, 'common/github_text.py', str(H / 'tasks_valid.jsonl'), str(H / 'meta_valid.jsonl'),
                    str(H / 'tasks_text.jsonl'), str(H / 'meta_text.jsonl')], check=True)

## 5. F2P/P2P and annotations (gold nodes, verified repro tests), 3 parallel shards

In [ ]:
def shards(k):
    return [valid[i::k] for i in range(k)]
procs = [subprocess.Popen([sys.executable, 'common/testmap.py', '--tasks', str(H / 'tasks_text.jsonl'), '--snapshots', str(H / 'snapshots'),
                           '--out', str(H / f'testmap_{i}.jsonl'), '--ids', *s]) for i, s in enumerate(shards(3)) if s]
assert all(p.wait() == 0 for p in procs)
with open(H / 'testmap.jsonl', 'w') as o:
    for f in sorted(H.glob('testmap_*.jsonl')): o.write(open(f).read())
procs = [subprocess.Popen([sys.executable, 'common/annotate.py', '--tasks', str(H / 'tasks_text.jsonl'), '--snapshots', str(H / 'snapshots'),
                           '--testmap', str(H / 'testmap.jsonl'), '--out', str(H / f'ann_{i}.jsonl'), '--verify', '--ids', *s])
         for i, s in enumerate(shards(3)) if s]
assert all(p.wait() == 0 for p in procs)
with open(H / 'annotations.jsonl', 'w') as o:
    for f in sorted(H.glob('ann_*.jsonl')): o.write(open(f).read())
print('annotated', sum(1 for _ in open(H / 'annotations.jsonl')))

## 6. Embedding map (fit on dev graphs + released embeddings), export, prompt-parity check

In [ ]:
EMB = man['embeddings']
pairs, seen = [], set()   # rule: the first dev graph (sorted by file name) of each dev repo that has embeddings
for g in sorted((COMP / 'graphs').glob('*.json')):
    repo = g.stem.split('_')[0]; e = COMP / 'embeddings' / f'{g.stem}.npz'
    if repo not in seen and e.exists(): pairs.append(f'{g}:{e}'); seen.add(repo)
mapfile = None
if EMB.get('map'):
    subprocess.run([sys.executable, 'pilot_d/embedding_parity.py', str(W / 'embedding_parity.json'), str(W / 'emb_cache'), *pairs],
                   check=True, env={**os.environ, 'PARITY_CANDIDATES': f"{EMB['model']}/{EMB['input']}"})
    mapfile = W / 'emb_cache' / f"W_{EMB['model']}_{EMB['input']}.npz"
cmd = [sys.executable, 'pilot_d/export_level_a.py', '--work', str(H), '--out', str(OUT), '--tasks', 'tasks_text.jsonl',
       '--meta', 'meta_text.jsonl', '--embed-model', EMB['model'], '--embed-input', EMB['input'],
       '--wheels-req', *sorted(glob.glob('pilot_d/env/*.txt'))]
if mapfile: cmd += ['--embed-map', str(mapfile)]
subprocess.run(cmd, check=True)
subprocess.run([sys.executable, 'pilot_d/check_prompt_parity.py', str(OUT), '--out', str(W / 'prompt_parity.json')], check=True)

## 7. Summary (aggregates + our own task ids; safe to commit)

In [ ]:
meta = {json.loads(l)['instance_id']: json.loads(l) for l in open(OUT / 'heldout_meta.jsonl')}
pp = json.load(open(W / 'prompt_parity.json'))['aggregate']
summary = {'export': json.load(open(OUT / 'EXPORT.json')), 'built': len(val), 'valid_kaggle': len(valid),
           'by_repo': {r: {'built': sum(v['repo'] == r for v in val), 'valid': sum(v['repo'] == r and v['valid'] for v in val)}
                       for r in sorted({v['repo'] for v in val})},
           'probe_eligible': sorted(i for i, m in meta.items() if m['probe']['eligible']),
           'issue_text': {i: m['problem_statement_source'] for i, m in meta.items()},
           'invalid': {v['instance_id']: ('empty_patch_passes' if v.get('base_resolved') else (v.get('gold_symptom') or v.get('gold_error') or 'gold_fails')[:80])
                       for v in val if not v['valid']},
           'prompt_parity': pp, 'test_deps_used': used,
           'embedding_parity': json.load(open(W / 'embedding_parity.json')) if (W / 'embedding_parity.json').exists() else None}
json.dump(summary, open(W / 'D0_summary.json', 'w'), indent=1)
print(json.dumps({k: summary[k] for k in ('built', 'valid_kaggle', 'by_repo', 'prompt_parity')}, indent=1))